# 20 · Deep learning et premiers pas en PyTorch — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch20_deep_learning/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch20_deep_learning/03_notebook.ipynb`, créée par `python tools/start_chapter.py 20`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 20.Q1 à 20.Q12, 20.R1 à 20.R3, 20.1 à 20.4, 20.6, 20.7, 20.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 20.13 | Premiers tenseurs PyTorch : créer, convertir, dtype, device | 📦 | ★ | 15 |
|  | 20.14 | Prédire les formes produites par les couches utilitaires | 🔮 | ★ | 15 |
|  | 20.15 | Autograd : retrouver les gradients du réseau minuscule du ch. 18 | 📦 | ★★ | 20 |
|  | 20.16 | dropout_forward et dropout_backward | 🔨 | ★★ | 25 |
|  | 20.17 | layer_norm, l2_penalty et EarlyStopping | 🔨 | ★★ | 30 |
| B | 20.18 | Écrire sa classe nn.Module et compter ses paramètres | 📦 | ★★ | 25 |
|  | 20.19 | Dataset et DataLoader : MNIST en mini-batches | 📦 | ★★ | 25 |
|  | 20.20 | Zoo des couches : convolution, pooling, récurrente, utilitaires, bruit | 📦 | ★★ | 30 |
|  | 20.21 | Reproduire la figure 20.17 : le plan de VGG16 couche par couche | 🎨 | ★★ | 30 |
|  | 20.22 | batchnorm_forward et batchnorm_backward | 🔨 | ★★★ | 75 |
| C | 20.23 | La boucle d'entraînement PyTorch complète sur MNIST | 📦 | ★★★ | 45 |
|  | 20.24 | Chasse aux bugs PyTorch : cinq erreurs classiques | 🐛 | ★★ | 30 |
|  | 20.25 | Courbes train/validation : overfitting, eval() oublié, fuite | 📈 | ★★ | 20 |
|  | 20.26 | Un modèle ImageNet face à des images qu'il n'a jamais vues | 🔮 | ★★ | 25 |
|  | 20.27 | Sauvegarder, recharger et reproduire un entraînement | 🛠️ | ★★ | 25 |
| D | 20.28 | Dropout, batchnorm, AdamW et early stopping sur Fashion-MNIST | 📦 | ★★★ | 60 |
|  | 20.29 | Ablation : qu'est-ce qui retarde l'overfitting ? | 🔬 | ★★★ | 60 |
|  | 20.30 | Un réseau de 30 couches : connexions résiduelles et LayerNorm | 🔬 | ★★★ | 45 |
|  | 20.31 | Défi Fashion-MNIST : au moins 90 % avec un MLP de 300 k paramètres au plus | 🏆 | ★★★★ | 120 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="20")  # your own library

## Objectifs et rappel express

- Prévoir la forme d'un tenseur couche après couche, et dimensionner la sortie d'un réseau.
- Calculer à la main un dropout inversé, une batchnorm (entraînement et évaluation) et une layer norm, puis les programmer dans `mylearn.nn.regularization` et les retrouver dans PyTorch.
- Écrire un `nn.Module`, un `DataLoader` et une boucle d'entraînement PyTorch, et mener une ablation de régularisation.

**Rappel express.** Dropout inversé : $\tilde a = a\,m/(1 - p)$, $m \sim \mathrm{Bernoulli}(1 - p)$, un nouveau masque à chaque passe avant, l'identité en évaluation. Batchnorm : $\hat x = (x - \mu_B)/\sqrt{\sigma_B^2 + \epsilon}$ (variance biaisée), $y = \gamma\hat x + \beta$ ; statistiques glissantes $\mu^{\text{run}} \leftarrow (1 - \rho)\,\mu^{\text{run}} + \rho\,\mu_B$ avec $\rho = 0{,}1$ et la variance non biaisée. Layer norm : la même formule, sur les features de chaque exemple. Pénalité L2 : $\frac{\lambda}{2}\sum\lVert\mathbf{W}\rVert^2$, de gradient $\lambda\mathbf{W}$. Boucle PyTorch : `model.train()`, puis pour chaque mini-batch `logits = model(X)`, `loss = loss_fn(logits, y)`, `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` ; `model.eval()` et `torch.no_grad()` pour mesurer.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 20.1 à 20.4, 20.6, 20.7 et 🧮 20.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une forme : entre guillemets (`"(3, 5)"`, ou `"(7,)"` pour une seule dimension). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les preuves ∂ 20.5 et 20.8, la réflexion (🗣️ 20.10, ⚖️ 20.11, 📄 20.12) et l'entretien se corrigent avec `05_solutions.md`.

**20.Q1 — Deep learning et machine learning : qui contient qui ?**

In [ ]:
# 20.Q1: the values you found on paper
answer_20_Q1a = ...  # a) True or False
answer_20_Q1b = ...  # b) True or False
answer_20_Q1c = ...  # c) the letter of your choice, e.g. "E"
answer_20_Q1d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_20_Q1a, answer_20_Q1b, answer_20_Q1c, answer_20_Q1d]):
    wb.check(f"20.Q1{letter}", answer)

**20.Q2 — Tenseurs : nombre de dimensions, taille, forme**

In [ ]:
# 20.Q2: the values you found on paper
answer_20_Q2a = ...  # a) a whole number
answer_20_Q2b = ...  # b) a whole number
answer_20_Q2c = ...  # c) a shape as a string, e.g. "(3, 5)"
answer_20_Q2d = ...  # d) True or False
answer_20_Q2e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_20_Q2a, answer_20_Q2b, answer_20_Q2c, answer_20_Q2d, answer_20_Q2e]):
    wb.check(f"20.Q2{letter}", answer)

**20.Q3 — Couche d'entrée, couche de sortie : laquelle contient des neurones ?**

In [ ]:
# 20.Q3: the values you found on paper
answer_20_Q3a = ...  # a) True or False
answer_20_Q3b = ...  # b) True or False
answer_20_Q3c = ...  # c) a whole number
answer_20_Q3d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_20_Q3a, answer_20_Q3b, answer_20_Q3c, answer_20_Q3d]):
    wb.check(f"20.Q3{letter}", answer)

**20.Q4 — Couche dense et couche d'activation séparée**

In [ ]:
# 20.Q4: the values you found on paper
answer_20_Q4a = ...  # a) a whole number
answer_20_Q4b = ...  # b) a whole number
answer_20_Q4c = ...  # c) True or False
answer_20_Q4d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_20_Q4a, answer_20_Q4b, answer_20_Q4c, answer_20_Q4d]):
    wb.check(f"20.Q4{letter}", answer)

**20.Q5 — Vrai ou faux : le dropout tire ses neurones une fois par epoch**

In [ ]:
# 20.Q5: the values you found on paper
answer_20_Q5a = ...  # a) True or False
answer_20_Q5b = ...  # b) a whole number
answer_20_Q5c = ...  # c) a number
answer_20_Q5d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_20_Q5a, answer_20_Q5b, answer_20_Q5c, answer_20_Q5d]):
    wb.check(f"20.Q5{letter}", answer)

**20.Q6 — Vrai ou faux : la batchnorm n'a aucun paramètre**

In [ ]:
# 20.Q6: the values you found on paper
answer_20_Q6a = ...  # a) True or False
answer_20_Q6b = ...  # b) a whole number
answer_20_Q6c = ...  # c) a whole number
answer_20_Q6d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_20_Q6a, answer_20_Q6b, answer_20_Q6c, answer_20_Q6d]):
    wb.check(f"20.Q6{letter}", answer)

**20.Q7 — Convolution et pooling en un coup d'œil**

In [ ]:
# 20.Q7: the values you found on paper
answer_20_Q7a = ...  # a) a whole number
answer_20_Q7b = ...  # b) a whole number
answer_20_Q7c = ...  # c) a whole number
answer_20_Q7d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_20_Q7a, answer_20_Q7b, answer_20_Q7c, answer_20_Q7d]):
    wb.check(f"20.Q7{letter}", answer)

**20.Q8 — Récurrence n'est pas récursion**

In [ ]:
# 20.Q8: the values you found on paper
answer_20_Q8a = ...  # a) True or False
answer_20_Q8b = ...  # b) the letter of your choice, e.g. "E"
answer_20_Q8c = ...  # c) True or False

for letter, answer in zip("abc", [answer_20_Q8a, answer_20_Q8b, answer_20_Q8c]):
    wb.check(f"20.Q8{letter}", answer)

**20.Q9 — Aplatir, remodeler, rogner, rembourrer, suréchantillonner, bruiter**

In [ ]:
# 20.Q9: the values you found on paper
answer_20_Q9a = ...  # a) a whole number
answer_20_Q9b = ...  # b) a whole number
answer_20_Q9c = ...  # c) a whole number
answer_20_Q9d = ...  # d) the letter of your choice, e.g. "E"
answer_20_Q9e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_20_Q9a, answer_20_Q9b, answer_20_Q9c, answer_20_Q9d, answer_20_Q9e]):
    wb.check(f"20.Q9{letter}", answer)

**20.Q10 — VGG16 : ce qu'il reconnaît et ce qu'il ignore**

In [ ]:
# 20.Q10: the values you found on paper
answer_20_Q10a = ...  # a) a whole number
answer_20_Q10b = ...  # b) a whole number
answer_20_Q10c = ...  # c) a whole number
answer_20_Q10d = ...  # d) the letter of your choice, e.g. "E"
answer_20_Q10e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_20_Q10a, answer_20_Q10b, answer_20_Q10c, answer_20_Q10d, answer_20_Q10e]):
    wb.check(f"20.Q10{letter}", answer)

**20.Q11 — Construire un réseau profond : par où commencer ?**

In [ ]:
# 20.Q11: the values you found on paper
answer_20_Q11a = ...  # a) the letter of your choice, e.g. "E"
answer_20_Q11b = ...  # b) the letter of your choice, e.g. "E"
answer_20_Q11c = ...  # c) True or False
answer_20_Q11d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_20_Q11a, answer_20_Q11b, answer_20_Q11c, answer_20_Q11d]):
    wb.check(f"20.Q11{letter}", answer)

**20.Q12 — Explication complète ou explication satisfaisante ?**

In [ ]:
# 20.Q12: the values you found on paper
answer_20_Q12a = ...  # a) True or False
answer_20_Q12b = ...  # b) the letter of your choice, e.g. "E"
answer_20_Q12c = ...  # c) the letter of your choice, e.g. "E"
answer_20_Q12d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_20_Q12a, answer_20_Q12b, answer_20_Q12c, answer_20_Q12d]):
    wb.check(f"20.Q12{letter}", answer)

**20.R1 — Ch. 19 : que fait weight_decay dans AdamW ?**

In [ ]:
# 20.R1: the values you found on paper
answer_20_R1a = ...  # a) a number
answer_20_R1b = ...  # b) True or False
answer_20_R1c = ...  # c) a number

for letter, answer in zip("abc", [answer_20_R1a, answer_20_R1b, answer_20_R1c]):
    wb.check(f"20.R1{letter}", answer)

**20.R2 — Ch. 17 : pourquoi calcule-t-on la loss à partir des logits ?**

In [ ]:
# 20.R2: the values you found on paper
answer_20_R2a = ...  # a) True or False
answer_20_R2b = ...  # b) a whole number
answer_20_R2c = ...  # c) a whole number
answer_20_R2d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_20_R2a, answer_20_R2b, answer_20_R2c, answer_20_R2d]):
    wb.check(f"20.R2{letter}", answer)

**20.R3 — Ch. 13 : limiter l'overfitting d'un arbre, et l'équivalent pour un réseau**

In [ ]:
# 20.R3: the values you found on paper
answer_20_R3a = ...  # a) the letter of your choice, e.g. "E"
answer_20_R3b = ...  # b) the letter of your choice, e.g. "E"
answer_20_R3c = ...  # c) True or False

for letter, answer in zip("abc", [answer_20_R3a, answer_20_R3b, answer_20_R3c]):
    wb.check(f"20.R3{letter}", answer)

**Ex 20.1 — Suivre la forme d'un tenseur couche après couche**

In [ ]:
# Ex 20.1: the values you found on paper
answer_20_1a = ...  # a) a whole number
answer_20_1b = ...  # b) a shape as a string, e.g. "(3, 5)"
answer_20_1c = ...  # c) a shape as a string, e.g. "(3, 5)"
answer_20_1d = ...  # d) a shape as a string, e.g. "(3, 5)"
answer_20_1e = ...  # e) a whole number
answer_20_1f = ...  # f) a shape as a string, e.g. "(3, 5)"
answer_20_1g = ...  # g) a shape as a string, e.g. "(3, 5)"
answer_20_1h = ...  # h) a shape as a string, e.g. "(3, 5)"

for letter, answer in zip("abcdefgh", [answer_20_1a, answer_20_1b, answer_20_1c, answer_20_1d, answer_20_1e, answer_20_1f, answer_20_1g, answer_20_1h]):
    wb.check(f"20.1{letter}", answer)

**Ex 20.2 — Dimensionner la couche de sortie et choisir la loss**

In [ ]:
# Ex 20.2: the values you found on paper
answer_20_2a = ...  # a) a whole number
answer_20_2b = ...  # b) the letter of your choice, e.g. "E"
answer_20_2c = ...  # c) a whole number
answer_20_2d = ...  # d) the letter of your choice, e.g. "E"
answer_20_2e = ...  # e) the letter of your choice, e.g. "E"
answer_20_2f = ...  # f) a whole number
answer_20_2g = ...  # g) the letter of your choice, e.g. "E"
answer_20_2h = ...  # h) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdefgh", [answer_20_2a, answer_20_2b, answer_20_2c, answer_20_2d, answer_20_2e, answer_20_2f, answer_20_2g, answer_20_2h]):
    wb.check(f"20.2{letter}", answer)

**Ex 20.3 — Dropout inversé à la main**

In [ ]:
# Ex 20.3: the values you found on paper
answer_20_3a = ...  # a) a whole number
answer_20_3b = ...  # b) a whole number
answer_20_3c = ...  # c) a number, 4 decimals
answer_20_3d = ...  # d) a number, 4 decimals
answer_20_3e = ...  # e) a number, 4 decimals
answer_20_3f = ...  # f) a number
answer_20_3g = ...  # g) a number, 4 decimals
answer_20_3h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_20_3a, answer_20_3b, answer_20_3c, answer_20_3d, answer_20_3e, answer_20_3f, answer_20_3g, answer_20_3h]):
    wb.check(f"20.3{letter}", answer)

**Ex 20.4 — Batchnorm à la main sur un mini-batch de quatre exemples**

In [ ]:
# Ex 20.4: the values you found on paper
answer_20_4a = ...  # a) a number
answer_20_4b = ...  # b) a number
answer_20_4c = ...  # c) a number
answer_20_4d = ...  # d) a number
answer_20_4e = ...  # e) a number, 4 decimals
answer_20_4f = ...  # f) a number, 4 decimals
answer_20_4g = ...  # g) True or False
answer_20_4h = ...  # h) a number
answer_20_4i = ...  # i) a number
answer_20_4j = ...  # j) a number, 4 decimals
answer_20_4k = ...  # k) a number, 3 decimals

for letter, answer in zip("abcdefghijk", [answer_20_4a, answer_20_4b, answer_20_4c, answer_20_4d, answer_20_4e, answer_20_4f, answer_20_4g, answer_20_4h, answer_20_4i, answer_20_4j, answer_20_4k]):
    wb.check(f"20.4{letter}", answer)

**Ex 20.6 — Compter les 138 millions de paramètres de VGG16**

In [ ]:
# Ex 20.6: the values you found on paper
answer_20_6a = ...  # a) a whole number
answer_20_6b = ...  # b) a whole number
answer_20_6c = ...  # c) a whole number
answer_20_6d = ...  # d) a whole number
answer_20_6e = ...  # e) a whole number
answer_20_6f = ...  # f) a number, 1 decimal
answer_20_6g = ...  # g) a number of MB, 1 decimal
answer_20_6h = ...  # h) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdefgh", [answer_20_6a, answer_20_6b, answer_20_6c, answer_20_6d, answer_20_6e, answer_20_6f, answer_20_6g, answer_20_6h]):
    wb.check(f"20.6{letter}", answer)

**Ex 20.7 — Connexion résiduelle et LayerNorm à la main**

In [ ]:
# Ex 20.7: the values you found on paper
answer_20_7a = ...  # a) a number
answer_20_7b = ...  # b) a number
answer_20_7c = ...  # c) a number, 4 decimals
answer_20_7d = ...  # d) a number
answer_20_7e = ...  # e) True or False
answer_20_7f = ...  # f) a number of 1e-9, 4 decimals
answer_20_7g = ...  # g) a whole number
answer_20_7h = ...  # h) a number, 4 decimals

for letter, answer in zip("abcdefgh", [answer_20_7a, answer_20_7b, answer_20_7c, answer_20_7d, answer_20_7e, answer_20_7f, answer_20_7g, answer_20_7h]):
    wb.check(f"20.7{letter}", answer)

**Ex 20.9 — Fermi : entraîner VGG16 sur ImageNet chez soi ?**

In [ ]:
# Ex 20.9: the values you found on paper
answer_20_9a = ...  # a) a number of billions, 1 decimal
answer_20_9b = ...  # b) a number of 1e18, 2 decimals
answer_20_9c = ...  # c) a number of days, 1 decimal
answer_20_9d = ...  # d) a whole number of days
answer_20_9e = ...  # e) a number of GB, 2 decimals

for letter, answer in zip("abcde", [answer_20_9a, answer_20_9b, answer_20_9c, answer_20_9d, answer_20_9e]):
    wb.check(f"20.9{letter}", answer)

## Partie A · Tenseurs, autograd, et ta librairie regularization.py

Tes premiers pas dans PyTorch : des tenseurs (20.13), les formes que produisent les couches utilitaires (20.14), l'autograd sur le réseau minuscule du ch. 18 (20.15) ; puis tu écris en NumPy, dans ta librairie, le dropout (20.16), la layer norm, la pénalité L2 et l'early stopping (20.17). La cellule suivante importe PyTorch et définit les outils des parties A à D. Ne recopie jamais un nombre à la main : chaque réponse vérifiée est calculée par ton code, et pas arrondie (sauf mention contraire).

In [ ]:
# Tools for the notebook exercises (parts A to D)
import contextlib
import re
import copy
import math
import tempfile
import time

import matplotlib.pyplot as plt
import numpy as np
from wb.errors import MylearnMissingError

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset, TensorDataset
except ImportError:                       # a computer without PyTorch (Mac Intel): this chapter is done on Colab
    torch = None
    print("⏳ PyTorch absent sur cet ordinateur : ce chapitre se fait sur Colab (00_setup/INSTALL_LOCAL.md). Les "
          "cellules qui utilisent PyTorch affichent ⏳ au lieu de s'exécuter ; tes fonctions NumPy (20.16, 20.17, "
          "20.22) se testent aussi dans un terminal : python -m pytest tests/test_ch20_regularization.py -q")

    def _skip_torch_cells(lines):
        """IPython input transformer: without PyTorch, a cell that uses it prints ⏳ instead of stopping "Run all"."""
        if re.search(r"\b(torch|torchvision)\b|(?<![\w.])nn\.|\b(DataLoader|TensorDataset|Dataset)\b", "".join(lines)):
            return ['print("⏳ PyTorch absent sur cet ordinateur : cette cellule se fait sur Colab.")\n']
        return lines

    try:
        _shell = get_ipython()
    except NameError:                     # not in a notebook
        _shell = None
    if _shell is not None and all(getattr(f, "__name__", "") != "_skip_torch_cells" for f in _shell.input_transformers_cleanup):
        _shell.input_transformers_cleanup.append(_skip_torch_cells)

DEVICE = cfg.device                       # "cuda" if a GPU is available, else "cpu" (wb.setup)


def verdict(ex_id, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message."""
    print(f"✅ Ex {ex_id} : {success}" if ok else f"❌ Ex {ex_id} : {failure}")


def shape_text(value):
    """A shape written as "[2, 3]", "2, 3", "torch.Size([2, 3])" or as a tuple, rewritten as the string "(2, 3)"."""
    if isinstance(value, (tuple, list) + ((torch.Size,) if torch is not None else ())) and all(isinstance(v, int) for v in value):
        return str(tuple(value))
    if isinstance(value, str) and re.fullmatch(r"\s*(torch\.Size)?[\s(\[]*\d+(\s*,\s*\d+)*\s*,?[\s)\]]*\s*", value):
        return str(tuple(int(n) for n in re.findall(r"\d+", value)))
    return value


def filled(*values):
    """True when none of the values is still `...` (or None): the answer has been written."""
    return all(value is not ... and value is not None for value in values)


class StopCheck(Exception):
    """Raised by pair() once its ❌ line is printed: the rest of the check would make no sense; checked() stops quietly."""


@contextlib.contextmanager
def checked(ex_id):
    """Like wb.attempt (⏳ while the code is not written yet), but any other error raised by YOUR code becomes one ❌
    line instead of stopping the cell, so that what follows can say more."""
    with wb.attempt(ex_id):
        try:
            yield
        except (NotImplementedError, ImportError, MylearnMissingError):
            raise
        except StopCheck:                                     # the reason is already printed
            pass
        except Exception as exc:                              # a bug in the learner's code: report it, go on
            if isinstance(exc, TypeError) and "NoneType" in str(exc):
                raise                                         # wb.attempt explains the forgotten return
            hint = " ; une fonction ou une méthode oublie-t-elle son return ?" if "NoneType" in str(exc) else ""
            print(f"❌ Ex {ex_id} : ton code lève {type(exc).__name__} : {str(exc)[:300]}{hint}")


def written(probe):
    """False while probe() stops on code not written yet (NotImplementedError, or no mylearn at all); True otherwise,
    even when it raises another error (the tests then say which)."""
    try:
        probe()
    except (NotImplementedError, ImportError, MylearnMissingError):
        return False
    except Exception:
        return True
    return True


def as_numpy(value):
    """A NumPy array (float64) from a tensor (detached, on the CPU), an array or a list; None if impossible."""
    if torch is not None and isinstance(value, torch.Tensor):
        return value.detach().cpu().double().numpy()
    try:
        return np.asarray(value, dtype=float)
    except (TypeError, ValueError):
        return None


def same(ex_id, label, got, want, rtol=1e-9, atol=1e-12):
    """✅ if `got` (array or tensor) equals `want` up to rounding, ❌ with both values otherwise; returns the verdict."""
    if got is None:
        print(f"❌ Ex {ex_id} : {label} renvoie None : as-tu oublié le return ?")
        return False
    arr = as_numpy(got)
    if arr is None:
        print(f"❌ Ex {ex_id} : {label} : attendu un tableau de nombres, reçu un objet {type(got).__name__}.")
        return False
    want = as_numpy(want)
    if arr.shape != want.shape:
        print(f"❌ Ex {ex_id} : {label} : attendu la forme {want.shape}, reçu la forme {arr.shape}.")
        return False
    ok = bool(np.allclose(arr, want, rtol=rtol, atol=atol, equal_nan=True))
    shown = np.round(arr, 8).tolist() if arr.size <= 6 else f"un tableau {arr.shape}"
    if ok or want.size <= 6:
        failure = f"{label} : attendu {np.round(want, 8).tolist()}, reçu {shown}."
    else:
        bad = np.argwhere(~np.isclose(arr, want, rtol=rtol, atol=atol, equal_nan=True))[0]
        at = tuple(int(i) for i in bad) if arr.ndim > 1 else int(bad[0])
        failure = (f"{label} : première différence à l'indice {at} : attendu {float(want[tuple(bad)]):.8g}, reçu "
                   f"{float(arr[tuple(bad)]):.8g} (plus grand écart {np.nanmax(np.abs(arr - want)):.3g}).")
    verdict(ex_id, ok, f"{label} : {shown}, comme attendu.", failure)
    return ok


def pair(ex_id, label, result, names=("out", "mask")):
    """The values of a result that must be a tuple (out, mask), (dx, dgamma, dbeta)...: otherwise one ❌ line, and the
    check stops there (StopCheck) instead of unpacking an array by mistake."""
    if result is None:
        print(f"❌ Ex {ex_id} : {label} renvoie None : as-tu oublié le return ?")
    elif not isinstance(result, (tuple, list)) or len(result) != len(names):
        what = (f"un {type(result).__name__} de {len(result)} valeurs" if isinstance(result, (tuple, list))
                else f"un {type(result).__name__}" + (f" de forme {tuple(np.shape(result))}" if hasattr(result, "shape") else ""))
        print(f"❌ Ex {ex_id} : {label} doit rendre le {'couple' if len(names) == 2 else 'triplet'} "
              f"({', '.join(names)}), reçu {what}.")
    else:
        return tuple(result)
    raise StopCheck


def refuses(ex_id, label, call, error=ValueError):
    """✅ if call() raises `error`, ❌ otherwise (the error raised, or nothing)."""
    try:
        call()
    except error as exc:
        print(f"✅ Ex {ex_id} : {label} : {error.__name__} levée ({str(exc)[:120]}).")
        return True
    except (NotImplementedError, MylearnMissingError):
        raise
    except Exception as exc:
        print(f"❌ Ex {ex_id} : {label} : attendu {error.__name__}, reçu {type(exc).__name__} : {str(exc)[:160]}")
        return False
    print(f"❌ Ex {ex_id} : {label} : attendu {error.__name__}, mais rien n'a été levé.")
    return False


def n_params(model):
    """The number of parameters (learned numbers) of a torch module."""
    return sum(p.numel() for p in model.parameters())


TEST_FILE = "tests/test_ch20_regularization.py"


def run_reg_tests(keyword, impl="learner"):
    """Run the tests of mylearn/nn/regularization.py selected by `keyword` (on YOUR code by default)."""
    command = [sys.executable, "-m", "pytest", TEST_FILE, "-k", keyword, "-q", "-p", "no:cacheprovider",
               "--color=no", "-rf", "--tb=no"]
    if impl != "learner":
        command.append(f"--impl={impl}")
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True,
                            env={**os.environ, "COLUMNS": "1000"})   # long lines: the reason of each failure
    lines = result.stdout.strip().splitlines()
    failed = [line for line in lines if line.startswith("FAILED ")]
    if any("'NoneType' object has no attribute" in line for line in failed):
        print("💡 un attribut est lu sur None : une méthode oublie-t-elle son return ?")
    for line in failed[:8]:                                  # the test, then the reason of its failure
        name, _, reason = line.removeprefix(f"FAILED {TEST_FILE}::").partition(" - ")
        print(f"❌ {name}\n   {reason[:800]}")
    if len(failed) > 8:
        print(f"   ... and {len(failed) - 8} other failed test(s)")
    print("pytest:", lines[-1] if lines else result.stderr.strip()[-300:])


print("tools ready · torch", torch.__version__ if torch is not None else "absent", "· device", DEVICE)

### Ex 20.13 — Premiers tenseurs PyTorch : créer, convertir, dtype, device 📦 ★ ⏱️ 15 min
**Objectif :** passer d'un tableau NumPy à un tenseur PyTorch de la bonne forme, du bon type et sur le bon appareil.  
**Prérequis :** 0A (tableaux NumPy, `reshape`) · fiche §20.2.1 (encadré « les tenseurs PyTorch ») · **Fil rouge :** MNIST · **Parcours :** R, C

`X_13` contient 64 images MNIST, un tableau NumPy d'entiers `uint8` de forme `(64, 28, 28)`, et `Y_13` leurs labels. Un réseau PyTorch attend des réels `float32`, dans l'ordre NCHW, et les labels d'une classification en `int64`. Construis :
- `images_13` : un tenseur `float32` de forme `(64, 1, 28, 28)` (un canal), les valeurs de `X_13` divisées par 255 ;
- `labels_13` : un tenseur `int64` de forme `(64,)`, les labels ;
- `images_dev_13` : `images_13` envoyé sur l'appareil `DEVICE` (le CPU, ou le GPU s'il y en a un).

Puis réponds :
a) `mean_13` : la valeur moyenne des pixels de `images_13`, en `float` Python (`.item()`) ;
b) `dtype_13` : le dtype de `torch.from_numpy(np.zeros(3))`, en chaîne (`str(...)`) ;
c) `shared_13` : avec `vec = np.zeros(3)`, `t = torch.from_numpy(vec)`, puis `vec[0] = 5.0`, est-ce que `t[0]` vaut 5 ? (`True` ou `False` ; essaie.)

Dans tes notes : que se passerait-il, dans c), avec `torch.tensor(vec)` à la place de `torch.from_numpy(vec)` ? Quand ce partage de mémoire est-il utile, et quand est-il dangereux ?

In [ ]:
X_13, Y_13 = wb.datasets.load_mnist("train", n=64)        # 64 images: NumPy, uint8 (64, 28, 28) and int64 (64,)
print("X_13:", X_13.shape, X_13.dtype, "· values from", X_13.min(), "to", X_13.max(), "· Y_13:", Y_13.shape, Y_13.dtype)

In [ ]:
images_13 = ...      # a float32 tensor of shape (64, 1, 28, 28): the images of X_13, divided by 255
labels_13 = ...      # an int64 tensor of shape (64,): the labels Y_13
images_dev_13 = ...  # images_13 on the device DEVICE
mean_13 = ...        # a) the mean pixel value of images_13, a Python float
dtype_13 = ...       # b) the dtype of torch.from_numpy(np.zeros(3)), as a string, e.g. "torch.int8"
shared_13 = ...      # c) True or False

In [ ]:
if not filled(images_13, labels_13, images_dev_13):
    print("⏳ Ex 20.13 : définis images_13, labels_13 et images_dev_13.")
else:
    ok_13 = isinstance(images_13, torch.Tensor)
    if not ok_13:
        print(f"❌ Ex 20.13 : images_13 doit être un tenseur PyTorch (torch.from_numpy, torch.tensor), reçu {type(images_13).__name__}.")
    else:
        verdict("20.13", tuple(images_13.shape) == (64, 1, 28, 28), "images_13 a la forme (64, 1, 28, 28).",
                f"images_13 a la forme {tuple(images_13.shape)} : attendu (64, 1, 28, 28), l'ordre NCHW avec 1 canal (unsqueeze, reshape).")
        verdict("20.13", images_13.dtype == torch.float32, "images_13 est en float32.",
                f"images_13 est en {images_13.dtype} : attendu torch.float32, le type des réels d'un réseau (.float()).")
        if tuple(images_13.shape) == (64, 1, 28, 28):
            verdict("20.13", torch.allclose(images_13.double().reshape(64, 28, 28), torch.from_numpy(X_13 / 255.0), atol=1e-6),
                    "ses valeurs sont celles de X_13 divisées par 255.",
                    f"ses valeurs vont de {images_13.min().item():.3g} à {images_13.max().item():.3g} : attendu X_13 / 255, entre 0 et 1.")
    if isinstance(labels_13, torch.Tensor):
        verdict("20.13", labels_13.dtype == torch.int64 and tuple(labels_13.shape) == (64,) and bool((labels_13.cpu() == torch.from_numpy(Y_13)).all()),
                "labels_13 : les 64 labels, en int64.",
                f"labels_13 : attendu les labels de Y_13, en int64 et de forme (64,) ; reçu {labels_13.dtype}, forme {tuple(labels_13.shape)}.")
    else:
        print(f"❌ Ex 20.13 : labels_13 doit être un tenseur PyTorch, reçu {type(labels_13).__name__}.")
    if isinstance(images_dev_13, torch.Tensor) and ok_13:
        verdict("20.13", images_dev_13.device.type == DEVICE and torch.equal(images_dev_13.cpu(), images_13.cpu()),
                f"images_dev_13 est sur l'appareil {DEVICE}, avec les mêmes valeurs.",
                f"images_dev_13 est sur {images_dev_13.device.type} : attendu {DEVICE} (images_13.to(DEVICE)), avec les mêmes valeurs.")
wb.check("20.13a", mean_13)
wb.check("20.13b", str(dtype_13) if isinstance(dtype_13, torch.dtype) else dtype_13)
wb.check("20.13c", shared_13)

### Ex 20.14 — Prédire les formes produites par les couches utilitaires 🔮 ★ ⏱️ 15 min
**Objectif :** prédire la forme de sortie de couches utilitaires dans des cas que la fiche ne détaille pas, puis la mesurer.  
**Prérequis :** Ex 20.13 · ✏️ 20.1 · fiche §20.4.6, §20.4.8 (tableau des couches utilitaires) · **Parcours :** C

Quatre tenseurs : `x4_14` de forme `(2, 3, 5, 5)` (2 images de 3 canaux, 5 × 5 pixels), `x3_14` de forme `(2, 3, 5)` (2 séquences de 3 canaux et 5 pas de temps), `x2_14` de forme `(2, 3)` et `x12_14` de forme `(2, 12)`. Prédis, **avant** d'exécuter quoi que ce soit, la forme de sortie (entre guillemets, comme `"(1, 2)"`) :
a) `pad_14` : `nn.ZeroPad2d(1)(x4_14)` ;
b) `ceil_14` : `nn.MaxPool2d(2, ceil_mode=True)(x4_14)` (avec `ceil_mode=True`, la taille est arrondie au-dessus) ;
c) `overlap_14` : `nn.MaxPool2d(2, stride=1)(x4_14)` (une fenêtre de 2 × 2 qui avance d'un pixel à la fois) ;
d) `flat2_14` : `nn.Flatten()(x2_14)` ;
e) `up1d_14` : `nn.Upsample(scale_factor=2)(x3_14)` ;
f) `unflat_14` : `nn.Unflatten(1, (3, 4))(x12_14)`.

Puis exécute l'expérience : elle affiche, pour chaque couche, la forme d'entrée, la forme de sortie et le nombre de paramètres, montre un canal rembourré de zéros, et vérifie tes prédictions.

In [ ]:
x4_14 = torch.arange(150.0).reshape(2, 3, 5, 5)   # a mini-batch of 2 images, 3 channels, 5 x 5 pixels (NCHW)
x3_14 = torch.arange(30.0).reshape(2, 3, 5)        # 2 sequences, 3 channels, 5 time steps (N, C, L)
x2_14 = torch.ones(2, 3)                           # 2 examples, 3 features
x12_14 = torch.ones(2, 12)                         # 2 examples, 12 features
print("x4_14", tuple(x4_14.shape), "· x3_14", tuple(x3_14.shape), "· x2_14", tuple(x2_14.shape), "· x12_14", tuple(x12_14.shape))

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
pad_14 = ...       # a) nn.ZeroPad2d(1)(x4_14).shape, as a string, e.g. "(1, 2)"
ceil_14 = ...      # b) nn.MaxPool2d(2, ceil_mode=True)(x4_14)
overlap_14 = ...   # c) nn.MaxPool2d(2, stride=1)(x4_14)
flat2_14 = ...     # d) nn.Flatten()(x2_14)
up1d_14 = ...      # e) nn.Upsample(scale_factor=2)(x3_14)
unflat_14 = ...    # f) nn.Unflatten(1, (3, 4))(x12_14)

In [ ]:
if any(answer is ... or answer is None for answer in [pad_14, ceil_14, overlap_14, flat2_14, up1d_14, unflat_14]):
    print("⏳ Ex 20.14 : écris d'abord tes six prédictions, puis relance cette cellule.")
else:
    LAYERS_14 = {"pad_14": (nn.ZeroPad2d(1), x4_14), "ceil_14": (nn.MaxPool2d(2, ceil_mode=True), x4_14),
                 "overlap_14": (nn.MaxPool2d(2, stride=1), x4_14), "flat2_14": (nn.Flatten(), x2_14),
                 "up1d_14": (nn.Upsample(scale_factor=2), x3_14), "unflat_14": (nn.Unflatten(1, (3, 4)), x12_14)}
    print("layer                                 input            output           parameters")
    for name_14, (layer_14, x_14) in LAYERS_14.items():
        print(f"{str(layer_14)[:36]:37} {str(tuple(x_14.shape)):16} {str(tuple(layer_14(x_14).shape)):16} {n_params(layer_14)}")
    print("\nZeroPad2d(1) on the first channel of the first image (5 x 5 -> 7 x 7):")
    print(nn.ZeroPad2d(1)(x4_14)[0, 0].int())
    for letter_14, value_14 in zip("abcdef", [pad_14, ceil_14, overlap_14, flat2_14, up1d_14, unflat_14]):
        wb.check(f"20.14{letter_14}", shape_text(value_14))

Dans tes notes : pour chaque prédiction ratée, quelle règle de la couche t'a manqué ? Comment aurais-tu pu la trouver dans la documentation de PyTorch (cherche, sur la page de la couche, les formes d'entrée et de sortie, *Shape*) ?

### Ex 20.15 — Autograd : retrouver les gradients du réseau minuscule du ch. 18 📦 ★★ ⏱️ 20 min
**Objectif :** faire calculer par PyTorch les gradients que tu as calculés à la main au ch. 18, et comprendre leur rangement.  
**Prérequis :** Ex 20.13 · ✏️ 18.4 · fiche, section PyTorch en pratique (l'autograd) et §20.4.1 · **Parcours :** R, C

Le réseau minuscule de ✏️ 18.4 : deux entrées, deux neurones cachés, deux sorties, sans biais ni activation ; l'exemple `X_15` $= (1, 2)$, son label `Y_15` $= (1, 0)$, les poids `W1_15` et `W2_15` (une ligne par neurone de départ) et la loss $L = \frac12\sum (\hat y - y)^2$.
- Écris `tiny_loss_15(w1, w2)`, qui rend cette loss pour deux tenseurs de poids, avec des opérations **torch** seulement (`@`, `**`, `.sum()`) : la vérification crée `w1` et `w2` avec `requires_grad=True`, appelle ta fonction puis `backward()`.
- Écris la classe `Tiny15(nn.Module)`, le même réseau avec deux couches `nn.Linear(2, 2, bias=False, dtype=torch.float64)`, rangées dans les attributs `hidden` et `out` ; la vérification y copie les poids de 18.4 (transposés : `nn.Linear` range ses poids en `(n_out, n_in)`).

La vérification contrôle trois gradients, calculés par ton code :
a) `w1.grad` (forme `(2, 2)`) ;
b) `w2.grad` ;
c) `hidden.weight.grad` de `Tiny15`.

Tu réponds toi-même à d) et e), avec ton code :
d) `twice_15` : on appelle deux fois de suite `tiny_loss_15(w1, w2).backward()` sans remettre les gradients à zéro ; que vaut alors `w1.grad[0, 0]` ?
e) `after_15` : à partir des gradients d'**un** `backward()`, un pas de `torch.optim.SGD([w1, w2], lr=0.1)` ; la nouvelle loss (4 décimales).

Compare a), b) et e) à tes réponses de 18.4 (g, f et i).

Dans tes notes : pourquoi le gradient de c) est-il la transposée de celui de a) ? Dans une boucle d'entraînement, où place-t-on `optimizer.zero_grad()`, et que se passe-t-il si on l'oublie ?

In [ ]:
X_15 = np.array([1.0, 2.0])                           # the example (X, Y) of ✏️ 18.4
Y_15 = np.array([1.0, 0.0])                           # its label
W1_15 = np.array([[0.5, 1.0], [-0.5, 0.25]])          # (n_in, n_out), one row per neuron of departure (ch. 16)
W2_15 = np.array([[1.0, -1.0], [0.5, 0.5]])
print("the tiny network of 18.4: X, Y -> A, B -> C, D, no bias, no activation; loss = 1/2 * sum((y_hat - y)**2)")

In [ ]:
def tiny_loss_15(w1, w2):
    """The loss 1/2 * sum((x @ w1 @ w2 - y)**2) of the tiny network for the example X_15, Y_15, as a 0-d torch tensor.

    w1, w2: float64 tensors of shape (2, 2) (they may require gradients). Torch operations only (no NumPy inside), so
    that autograd can follow every operation.
    """
    raise NotImplementedError("tiny_loss_15")


class Tiny15(nn.Module):
    """The same network with two layers nn.Linear(2, 2, bias=False, dtype=torch.float64), named hidden and out."""

    def __init__(self):
        super().__init__()
        raise NotImplementedError("Tiny15.__init__")

    def forward(self, x):
        raise NotImplementedError("Tiny15.forward")


twice_15 = ...   # d) w1.grad[0, 0] after two calls of backward() without zeroing (a number)
after_15 = ...   # e) the loss after one step of torch.optim.SGD([w1, w2], lr=0.1), 4 decimals

In [ ]:
grad1_15 = grad2_15 = lin_grad_15 = None
with checked("20.15"):
    w1_15 = torch.tensor(W1_15, requires_grad=True)          # float64 leaves of the graph
    w2_15 = torch.tensor(W2_15, requires_grad=True)
    loss_15 = tiny_loss_15(w1_15, w2_15)
    if not isinstance(loss_15, torch.Tensor) or loss_15.ndim != 0:
        print(f"❌ Ex 20.15 : tiny_loss_15 doit rendre un tenseur 0-D (un seul nombre), reçu {type(loss_15).__name__}"
              + (f" de forme {tuple(loss_15.shape)}" if isinstance(loss_15, torch.Tensor) else "") + " (.sum() à la fin).")
    elif loss_15.grad_fn is None:
        print("❌ Ex 20.15 : la loss n'a pas de grad_fn : autograd ne peut rien remonter (calcule-la avec des opérations "
              "torch sur w1 et w2, sans passer par NumPy ni .item()).")
    else:
        print(f"loss = {loss_15.item():.4f} (the loss of 18.4 c)")
        loss_15.backward()
        grad1_15, grad2_15 = w1_15.grad, w2_15.grad
with checked("20.15"):
    tiny_15 = Tiny15()
    with torch.no_grad():                                    # nn.Linear stores (n_out, n_in): copy the TRANSPOSED weights
        tiny_15.hidden.weight.copy_(torch.tensor(W1_15.T))
        tiny_15.out.weight.copy_(torch.tensor(W2_15.T))
    out_15 = tiny_15(torch.tensor(X_15))
    verdict("20.15", tuple(out_15.shape) == (2,) and np.allclose(as_numpy(out_15), X_15 @ W1_15 @ W2_15),
            "Tiny15 calcule les mêmes sorties que x @ W1 @ W2.",
            f"Tiny15 rend {as_numpy(out_15)} : attendu les sorties de x @ W1 @ W2 (hidden, puis out, sans activation).")
    (0.5 * ((out_15 - torch.tensor(Y_15)) ** 2).sum()).backward()
    lin_grad_15 = tiny_15.hidden.weight.grad
wb.check("20.15a", as_numpy(grad1_15) if grad1_15 is not None else ..., computed=True)
wb.check("20.15b", as_numpy(grad2_15) if grad2_15 is not None else ..., computed=True)
wb.check("20.15c", as_numpy(lin_grad_15) if lin_grad_15 is not None else ..., computed=True)
wb.check("20.15d", twice_15)
wb.check("20.15e", after_15)

### Ex 20.16 — dropout_forward et dropout_backward 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire le dropout inversé en NumPy, passe avant et passe arrière, aux conventions de PyTorch.  
**Prérequis :** ch. 18 (passe arrière d'une couche) · ✏️ 20.3 · fiche §20.4.3 · **Fil rouge :** synthétique · **mylearn :** `nn/regularization.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/nn/regularization.py` (créé par `python tools/start_chapter.py 20`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy et la bibliothèque standard seulement : PyTorch est l'**oracle** des tests (sans PyTorch, ils comparent à une transcription NumPy des formules de la fiche, et aucun n'est ignoré). La cellule de vérification recharge ta librairie, essaie les exemples des docstrings et quelques cas choisis (comparés à PyTorch), puis lance les tests de l'exercice ; `python -m pytest tests/test_ch20_regularization.py -q` les lance tous dans un terminal.

**Contrat commun** (les docstrings, plus ces précisions que vérifient les tests) : tes fonctions acceptent des listes comme des tableaux ; elles ne modifient **jamais** leurs entrées (sauf, dans `batchnorm_forward` en entraînement, `running_mean` et `running_var`, mis à jour **en place**, et les tableaux passés à `EarlyStopping.restore`) ; les tableaux qu'elles rendent sont de **nouveaux** tableaux `float64` (jamais l'entrée elle-même, ni une vue) ; les entrées invalides lèvent `ValueError`, sans diffusion (*broadcasting*) : une forme différente est refusée même si NumPy saurait la diffuser.

Écris `dropout_forward(x, p=0.5, training=True, rng=None)` et `dropout_backward(dout, mask)`. **Contrat** propre au dropout :
- un masque **tiré valeur par valeur** (de la forme de `x`, chaque valeur gardée avec la probabilité $1 - p$, indépendamment des autres), avec le générateur `rng` reçu (`np.random.default_rng()` si `rng` vaut `None`), et un **nouveau** masque à chaque appel ; ses valeurs : 0 ou $1/(1 - p)$, et `out = x * mask` ;
- en évaluation (`training=False`) ou pour `p = 0` : `out` est une **copie** de `x` et `mask` vaut `None` ; pour `p = 1` : `out` et `mask` sont des zéros, sans division par 0 ; un `p` hors de $[0, 1]$ lève `ValueError` ;
- `dropout_backward` rend `dout * mask` (une copie de `dout` si `mask` vaut `None`) et refuse un masque d'une autre forme que `dout`.

La vérification essaie l'exemple de la docstring, la part de zéros et la moyenne sur 100 000 valeurs, deux appels avec le même générateur, l'évaluation, `p = 1`, un `p` invalide, la passe arrière contre l'autograd de PyTorch, puis lance les tests (`-k "test_dropout_"`).

Dans tes notes : pourquoi la passe arrière doit-elle utiliser le **même** masque que la passe avant ? Que se passerait-il si `dropout_backward` tirait un nouveau masque ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="20")   # reload your saved file
with checked("20.16"):
    R = mylearn.nn.regularization
    out_16, mask_16 = pair("20.16", "dropout_forward", R.dropout_forward(np.ones((2, 4)), p=0.5, rng=np.random.default_rng(0)))
    verdict("20.16", set(np.unique(out_16).tolist()) <= {0.0, 2.0} and np.array_equal(out_16, np.ones((2, 4)) * mask_16),
            "l'exemple de la docstring : des 0 et des 2, et out = x * mask.",
            f"l'exemple de la docstring : attendu des 0 et des 2 = 1/(1 - 0,5), avec out = x * mask ; reçu {np.unique(np.asarray(out_16)).tolist()}.")
    big_16, _ = pair("20.16", "dropout_forward", R.dropout_forward(np.ones(100_000), p=0.3, rng=np.random.default_rng(1)))
    zeros_16 = float(np.mean(np.asarray(big_16) == 0))
    verdict("20.16", abs(zeros_16 - 0.3) < 0.01 and abs(float(np.mean(big_16)) - 1.0) < 0.02,
            f"p = 0,3 sur 100 000 valeurs : {zeros_16:.3f} de zéros, une moyenne de {float(np.mean(big_16)):.3f}.",
            f"p = 0,3 : {zeros_16:.3f} de zéros et une moyenne de {float(np.mean(big_16)):.3f} ; attendu environ 0,3 de "
            "zéros et une moyenne de 1 (garde une valeur avec la probabilité 1 - p, et divise par 1 - p).")
    rng_16 = np.random.default_rng(2)
    first_16 = pair("20.16", "dropout_forward", R.dropout_forward(np.ones((5, 6)), p=0.5, rng=rng_16))[1]
    second_16 = pair("20.16", "dropout_forward", R.dropout_forward(np.ones((5, 6)), p=0.5, rng=rng_16))[1]
    verdict("20.16", not np.array_equal(first_16, second_16), "deux appels avec le même générateur tirent deux masques différents.",
            "deux appels avec le même générateur ont rendu le même masque : il faut un NOUVEAU masque à chaque passe avant.")
    x_16 = np.array([[1.0, -2.0, 3.0]])
    out_eval_16, mask_eval_16 = pair("20.16", "dropout_forward", R.dropout_forward(x_16, p=0.5, training=False))
    verdict("20.16", mask_eval_16 is None and np.array_equal(out_eval_16, x_16) and out_eval_16 is not x_16,
            "évaluation : une copie de x, et pas de masque.",
            f"évaluation : attendu (une copie de x, None), reçu ({out_eval_16!r}, {mask_eval_16!r}).")
    out_one_16, mask_one_16 = pair("20.16", "dropout_forward", R.dropout_forward(x_16, p=1.0, rng=np.random.default_rng(3)))
    verdict("20.16", np.array_equal(out_one_16, np.zeros((1, 3))) and np.array_equal(mask_one_16, np.zeros((1, 3))),
            "p = 1 : des zéros, sans division par 1 - p = 0.", f"p = 1 : attendu des zéros (et un masque de zéros), reçu {out_one_16!r}.")
    refuses("20.16", "dropout_forward avec p = 1,5", lambda: R.dropout_forward(x_16, p=1.5))
    x_t16 = torch.tensor([[0.5, -1.0, 2.0, 3.0]], dtype=torch.float64, requires_grad=True)
    out_t16, mask_t16 = pair("20.16", "dropout_forward", R.dropout_forward(x_t16.detach().numpy(), p=0.4, rng=np.random.default_rng(4)))
    dout_16 = np.array([[1.0, 2.0, -1.0, 0.5]])
    (x_t16 * torch.tensor(mask_t16)).backward(torch.tensor(dout_16))       # autograd through x * mask, the same mask
    same("20.16", "dropout_backward contre l'autograd de PyTorch (x * mask)", R.dropout_backward(dout_16, mask_t16), x_t16.grad)
    refuses("20.16", "dropout_backward avec un masque (1, 4) pour un dout (2, 4)", lambda: R.dropout_backward(np.ones((2, 4)), np.ones((1, 4))))
if written(lambda: mylearn.nn.regularization.dropout_forward(np.ones(2), p=0.5)):
    run_reg_tests("test_dropout_")
else:
    print("⏳ Ex 20.16 : les tests attendent tes fonctions.")

### Ex 20.17 — layer_norm, l2_penalty et EarlyStopping 🔨 ★★ ⏱️ 30 min
**Objectif :** écrire la layer norm, la pénalité L2 et un early stopping réutilisable avec ou sans PyTorch.  
**Prérequis :** ch. 9 (early stopping, régularisation L2) · ch. 18 · ✏️ 20.7 · fiche §20.4.8, section sur la régularisation · **Fil rouge :** synthétique · **mylearn :** `nn/regularization.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/nn/regularization.py` (créé par `python tools/start_chapter.py 20`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy et la bibliothèque standard seulement : PyTorch est l'**oracle** des tests (sans PyTorch, ils comparent à une transcription NumPy des formules de la fiche, et aucun n'est ignoré). La cellule de vérification recharge ta librairie, essaie les exemples des docstrings et quelques cas choisis (comparés à PyTorch), puis lance les tests de l'exercice ; `python -m pytest tests/test_ch20_regularization.py -q` les lance tous dans un terminal.

**Contrat commun** (les docstrings, plus ces précisions que vérifient les tests) : tes fonctions acceptent des listes comme des tableaux ; elles ne modifient **jamais** leurs entrées (sauf, dans `batchnorm_forward` en entraînement, `running_mean` et `running_var`, mis à jour **en place**, et les tableaux passés à `EarlyStopping.restore`) ; les tableaux qu'elles rendent sont de **nouveaux** tableaux `float64` (jamais l'entrée elle-même, ni une vue) ; les entrées invalides lèvent `ValueError`, sans diffusion (*broadcasting*) : une forme différente est refusée même si NumPy saurait la diffuser.

Écris `layer_norm(x, gamma=None, beta=None, eps=1e-5)`, `l2_penalty(weights, lam)` et la méthode `step` (puis `restore`) de la classe `EarlyStopping` (son `__init__` est fourni). **Contrat** propre à ces trois objets :
- `layer_norm` normalise sur le **dernier** axe de chaque exemple, avec la variance **biaisée** et `eps` sous la racine, puis applique `gamma` et `beta` (de forme `(D,)`, `D` la taille du dernier axe ; une autre forme, même diffusable comme `(1,)`, lève `ValueError`) ;
- `l2_penalty` rend le couple `(penalty, grads)` : la pénalité $\frac{\lambda}{2}\sum\lVert\mathbf{W}\rVert^2$ en `float` Python, et la liste des gradients $\lambda\mathbf{W}$, de nouveaux tableaux ; un `lam` négatif lève `ValueError` ;
- `EarlyStopping.step(value, params=None)` rend un `bool` Python (`True` quand il faut s'arrêter) : une amélioration est **stricte** (en mode `"min"`, `value < best_ - min_delta` ; en mode `"max"`, `value > best_ + min_delta`) ; la **première valeur finie** est toujours une amélioration, et un `nan` ne l'est jamais ; à chaque amélioration, `best_`, `best_epoch_` (le numéro du contrôle, à partir de 0) et `improved_` sont mis à jour, `counter_` repart à 0, et, si `params` est donné et `restore_best` vrai, une **copie profonde** des paramètres est gardée dans `best_params_` ; sinon `counter_` augmente de 1 ; l'arrêt a lieu quand `counter_` atteint `patience` (`stopped_` passe alors à `True`) ;
- `restore(params)` recopie les meilleurs paramètres **dans** les tableaux reçus (`p[...] = saved`), et lève `ValueError` si rien n'a été gardé ou si les formes ne correspondent pas.

La vérification essaie `layer_norm` contre `torch.nn.functional.layer_norm`, l'exemple de `l2_penalty`, la suite de la docstring d'`EarlyStopping` et une restauration après un changement des poids, puis lance les tests (`-k "test_layer_norm_ or test_l2_penalty_ or test_early_stopping_"`).

Dans tes notes : `EarlyStopping` ne connaît ni NumPy ni PyTorch ; avec un modèle PyTorch, à quoi sert l'attribut `improved_` (pense à `model.state_dict()`) ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="20")   # reload your saved file
with checked("20.17"):
    R = mylearn.nn.regularization
    x_17 = np.array([[1.0, 2.0, 3.0], [2.0, 0.0, 7.0]])
    gamma_17, beta_17 = np.array([1.0, 2.0, 0.5]), np.array([0.0, -1.0, 1.0])
    want_17 = torch.nn.functional.layer_norm(torch.tensor(x_17), (3,), torch.tensor(gamma_17), torch.tensor(beta_17), 1e-5)
    same("20.17", "layer_norm contre torch.nn.functional.layer_norm (gamma et beta)", R.layer_norm(x_17, gamma_17, beta_17), want_17)
    same("20.17", "layer_norm sans gamma ni beta, sur un tenseur 3-D", R.layer_norm(np.arange(24.0).reshape(2, 3, 4)),
         torch.nn.functional.layer_norm(torch.arange(24.0, dtype=torch.float64).reshape(2, 3, 4), (4,)))
    penalty_17, grads_17 = R.l2_penalty([np.array([[1.0, 2.0]]), np.array([3.0])], lam=0.5)
    verdict("20.17", isinstance(penalty_17, float) and abs(penalty_17 - 3.5) < 1e-12,
            "l2_penalty : (0,5 / 2) x (1 + 4 + 9) = 3,5, un float Python.",
            f"l2_penalty rend {penalty_17!r} : attendu 3,5 = (lam / 2) x la somme des carrés, un float Python.")
    same("20.17", "les gradients de la pénalité, lam x W", np.concatenate([np.ravel(g) for g in grads_17]), [0.5, 1.0, 1.5])
    stopper_17 = R.EarlyStopping(patience=2)
    stops_17 = [stopper_17.step(v) for v in [1.0, 0.8, 0.9, 0.85]]
    verdict("20.17", stops_17 == [False, False, False, True] and (stopper_17.best_, stopper_17.best_epoch_, stopper_17.counter_) == (0.8, 1, 2),
            "EarlyStopping(patience=2) sur 1,0 ; 0,8 ; 0,9 ; 0,85 : arrêt au 4ᵉ contrôle, meilleur 0,8 au contrôle 1.",
            f"EarlyStopping(patience=2) sur 1,0 ; 0,8 ; 0,9 ; 0,85 : step rend {stops_17}, et (best_, best_epoch_, counter_) = "
            f"{(stopper_17.best_, stopper_17.best_epoch_, stopper_17.counter_)} ; attendu [False, False, False, True] et (0.8, 1, 2).")
    w_17 = np.array([1.0, 2.0])
    keeper_17 = R.EarlyStopping(patience=3)
    keeper_17.step(0.5, params=[w_17])
    w_17 += 10.0                                          # the training goes on and changes the weights in place
    keeper_17.step(0.7, params=[w_17])
    keeper_17.restore([w_17])
    verdict("20.17", np.array_equal(w_17, [1.0, 2.0]), "restore remet en place les poids du meilleur contrôle (une vraie copie gardée).",
            f"après restore, les poids valent {w_17.tolist()} : attendu [1.0, 2.0], ceux du meilleur contrôle (garde une COPIE, "
            "remets-la dans les mêmes tableaux).")
if written(lambda: mylearn.nn.regularization.layer_norm(np.ones((1, 2)))):
    run_reg_tests("test_layer_norm_ or test_l2_penalty_ or test_early_stopping_")
else:
    print("⏳ Ex 20.17 : les tests attendent tes fonctions.")

## Partie B · Des modules, des données, le zoo des couches, et ta batchnorm

Écrire ta première classe `nn.Module` (20.18), un `Dataset` et un `DataLoader` (20.19), essayer les couches du tour d'horizon du livre (20.20), reconstruire VGG16 et dessiner son plan (20.21), puis écrire la batchnorm de ta librairie, passe avant et passe arrière (20.22).

### Ex 20.18 — Écrire sa classe nn.Module et compter ses paramètres 📦 ★★ ⏱️ 25 min
**Objectif :** écrire un perceptron multicouche comme une classe nn.Module, et compter ses paramètres.  
**Prérequis :** Ex 20.15 · ch. 16 (compter les paramètres d'une couche dense) · 0A (classes, héritage) · fiche §20.4.1, §20.4.2, section PyTorch en pratique · **Fil rouge :** MNIST · **Parcours :** R, C

Écris la classe `MLP18(nn.Module)` : son `__init__(self, sizes)` appelle `super().__init__()`, puis range dans `self.layers` un `nn.ModuleList` de couches `nn.Linear` de tailles `sizes[0] -> sizes[1] -> ... -> sizes[-1]` ; sa méthode `forward(x)` aplatit d'abord l'entrée (`nn.Flatten()`, qui garde la dimension des exemples), puis applique les couches, avec une ReLU après chacune **sauf la dernière**, qui rend les logits. Puis compte, avec ton code (`n_params`, fourni, ou `sum(p.numel() for p in model.parameters())`) :
a) `params_222_18` : les paramètres de `MLP18([2, 2, 2])`, le premier réseau de la figure 20.15 du livre ;
b) `params_2432_18` : ceux de `MLP18([2, 4, 3, 2])`, le réseau de la figure 20.16 ;
c) `params_mnist_18` : ceux de `MLP18([784, 128, 64, 10])`, un classifieur MNIST.

La vérification contrôle l'héritage, le `nn.ModuleList`, les tailles des couches, la forme des sorties sur un mini-batch MNIST `(64, 1, 28, 28)`, l'absence de softmax et l'ordre des couches, puis tes trois nombres.

Dans tes notes : que se passe-t-il si tu ranges les couches dans une liste Python ordinaire (`self.layers = [...]`) au lieu d'un `nn.ModuleList` ? Essaie, et regarde `len(list(model.parameters()))`. Pourquoi est-ce grave pour l'entraînement ?

In [ ]:
X_18, Y_18 = wb.datasets.load_mnist("train", n=64, normalize=True)          # float32 in [0, 1], (64, 28, 28)
BATCH_18 = torch.from_numpy(X_18).unsqueeze(1)                              # (64, 1, 28, 28)
print("a mini-batch of MNIST images:", tuple(BATCH_18.shape), BATCH_18.dtype)

In [ ]:
class MLP18(nn.Module):
    """A multilayer perceptron: nn.Flatten(), then nn.Linear layers of sizes[0] -> sizes[1] -> ... -> sizes[-1], with a
    ReLU after every layer except the last one (the last one returns the logits).

    The layers are stored in self.layers, an nn.ModuleList of nn.Linear (so that model.parameters() finds them).
    """

    def __init__(self, sizes):
        super().__init__()
        raise NotImplementedError("MLP18.__init__")

    def forward(self, x):
        raise NotImplementedError("MLP18.forward")


params_222_18 = ...    # a) the number of parameters of MLP18([2, 2, 2]) (an integer computed by your code)
params_2432_18 = ...   # b) the same for MLP18([2, 4, 3, 2])
params_mnist_18 = ...  # c) the same for MLP18([784, 128, 64, 10])

In [ ]:
with checked("20.18"):
    torch.manual_seed(18)
    model_18 = MLP18([784, 128, 64, 10])
    verdict("20.18", isinstance(model_18, nn.Module), "MLP18 est un nn.Module.", "MLP18 doit hériter de nn.Module (class MLP18(nn.Module)).")
    layers_18 = getattr(model_18, "layers", None)
    if not isinstance(layers_18, nn.ModuleList):
        print(f"❌ Ex 20.18 : self.layers doit être un nn.ModuleList, reçu {type(layers_18).__name__} (une liste Python "
              "ordinaire cache les couches à model.parameters() : l'optimiseur ne les verrait pas).")
    else:
        kinds_18 = [type(layer).__name__ for layer in layers_18]
        verdict("20.18", kinds_18 == ["Linear"] * 3 and [(l.in_features, l.out_features) for l in layers_18] == [(784, 128), (128, 64), (64, 10)],
                "self.layers : trois nn.Linear, 784 -> 128 -> 64 -> 10.",
                f"self.layers contient {kinds_18}, de tailles {[(getattr(l, 'in_features', '?'), getattr(l, 'out_features', '?')) for l in layers_18]} : attendu trois nn.Linear, 784 -> 128 -> 64 -> 10.")
        verdict("20.18", len(list(model_18.parameters())) == 6, "model.parameters() trouve les 6 tenseurs (3 poids, 3 biais).",
                f"model.parameters() trouve {len(list(model_18.parameters()))} tenseurs : attendu 6 (un poids et un biais par couche).")
    logits_18 = model_18(BATCH_18)
    if not isinstance(logits_18, torch.Tensor) or tuple(logits_18.shape) != (64, 10):
        print(f"❌ Ex 20.18 : model(BATCH_18) doit rendre un tenseur (64, 10), reçu {tuple(getattr(logits_18, 'shape', ()))} "
              "(nn.Flatten() d'abord : (64, 1, 28, 28) -> (64, 784)).")
    else:
        sums_18 = logits_18.detach().sum(dim=1)
        if torch.allclose(sums_18, torch.ones(64), atol=1e-4) and bool((logits_18 >= 0).all()):
            verdict("20.18", False, "", "les sorties sont positives et chaque ligne somme à 1 : c'est une softmax ; le modèle doit rendre des logits "
                    "(CrossEntropyLoss fera la softmax).")
        elif bool((logits_18 >= 0).all()):
            verdict("20.18", False, "", "aucune sortie n'est négative : y a-t-il une ReLU après la dernière couche ? Le modèle doit rendre des logits, "
                    "qui peuvent être négatifs.")
        else:
            verdict("20.18", True, "le modèle rend des logits (pas de softmax ni de ReLU à la fin).", "")
        first_18 = torch.relu(model_18.layers[0](BATCH_18.reshape(64, -1)))
        second_18 = model_18.layers[2](torch.relu(model_18.layers[1](first_18)))
        verdict("20.18", torch.allclose(second_18, logits_18, atol=1e-6), "la passe avant : ReLU après chaque couche sauf la dernière.",
                "la passe avant ne donne pas Linear -> ReLU -> Linear -> ReLU -> Linear : vérifie l'ordre, et qu'il n'y a pas de ReLU après la dernière couche.")
wb.check("20.18a", params_222_18)
wb.check("20.18b", params_2432_18)
wb.check("20.18c", params_mnist_18)

### Ex 20.19 — Dataset et DataLoader : MNIST en mini-batches 📦 ★★ ⏱️ 25 min
**Objectif :** écrire un Dataset et le parcourir en mini-batches mélangés et reproductibles.  
**Prérequis :** Ex 20.13 · 0A (méthodes spéciales `__len__` et `__getitem__`) · fiche §20.3.1, section PyTorch en pratique · **Fil rouge :** MNIST · **Parcours :** R, C

Écris la classe `MNISTData19(Dataset)`, qui garde 1 000 images MNIST (`X_19`, en `uint8`) et leurs labels : `len(dataset)` rend le nombre d'images, et `dataset[i]` le couple `(image, label)`, où `image` est un tenseur `float32` de forme `(1, 28, 28)` (les pixels divisés par 255) et `label` un tenseur 0-D `int64`. Écris aussi `make_loader_19(dataset, seed, drop_last=False)`, qui rend un `DataLoader` de mini-batches de 64, **mélangés à chaque epoch** par un générateur `torch.Generator().manual_seed(seed)`. Puis, avec ton code :
a) `n_batches_19` : le nombre de mini-batches d'une epoch sur les 1 000 images ;
b) `last_19` : le nombre d'images du dernier mini-batch ;
c) `dropped_19` : le nombre de mini-batches avec `drop_last=True`.

La vérification contrôle `len`, un exemple (forme, type, valeurs, label), un mini-batch (le `DataLoader` empile les exemples), deux epochs (mêmes labels, ordres différents) et deux chargeurs de même graine (même ordre), puis tes trois nombres.

Dans tes notes : pourquoi convertir l'image dans `__getitem__` plutôt qu'une fois pour toutes dans `__init__` ? Qu'est-ce qui change pour 60 000 images ? Et pour un dataset de photos lues sur le disque ?

In [ ]:
X_19, Y_19 = wb.datasets.load_mnist("train", n=1000)                      # NumPy: uint8 (1000, 28, 28), int64 (1000,)
print("X_19:", X_19.shape, X_19.dtype, "· Y_19:", Y_19.shape, Y_19.dtype)

In [ ]:
class MNISTData19(Dataset):
    """A Dataset of MNIST images: item i is the pair (image, label), where image is a float32 tensor of shape
    (1, 28, 28) with values in [0, 1] (the uint8 pixels divided by 255) and label a 0-d int64 tensor."""

    def __init__(self, images, labels):
        raise NotImplementedError("MNISTData19.__init__")

    def __len__(self):
        raise NotImplementedError("MNISTData19.__len__")

    def __getitem__(self, i):
        raise NotImplementedError("MNISTData19.__getitem__")


def make_loader_19(dataset, seed, drop_last=False):
    """A DataLoader over dataset: mini-batches of 64, shuffled at every epoch by a torch.Generator seeded with seed."""
    raise NotImplementedError("make_loader_19")


n_batches_19 = ...   # a) the number of mini-batches of one epoch over the 1000 images
last_19 = ...        # b) the number of images of the last mini-batch
dropped_19 = ...     # c) the number of mini-batches with drop_last=True

In [ ]:
with checked("20.19"):
    data_19 = MNISTData19(X_19, Y_19)
    verdict("20.19", isinstance(data_19, Dataset), "MNISTData19 est un Dataset.", "MNISTData19 doit hériter de torch.utils.data.Dataset.")
    verdict("20.19", len(data_19) == 1000, "len(dataset) = 1000.", f"len(dataset) vaut {len(data_19)} : attendu 1000, le nombre d'images.")
    item_19 = data_19[7]
    if not (isinstance(item_19, tuple) and len(item_19) == 2 and all(isinstance(t, torch.Tensor) for t in item_19)):
        print(f"❌ Ex 20.19 : dataset[i] doit rendre le couple (image, label) de deux tenseurs, reçu {type(item_19).__name__}.")
    else:
        image_19, label_19 = item_19
        verdict("20.19", tuple(image_19.shape) == (1, 28, 28) and image_19.dtype == torch.float32
                and torch.allclose(image_19[0].double(), torch.from_numpy(X_19[7] / 255.0), atol=1e-6),
                "dataset[7] : une image float32 (1, 28, 28), les pixels divisés par 255.",
                f"dataset[7] : attendu une image float32 de forme (1, 28, 28) et de valeurs X_19[7] / 255 ; reçu {image_19.dtype} de forme {tuple(image_19.shape)}, "
                f"de valeurs entre {float(image_19.min()):g} et {float(image_19.max()):g}.")
        verdict("20.19", label_19.dtype == torch.int64 and label_19.ndim == 0 and label_19.item() == int(Y_19[7]),
                "dataset[7] : son label, un tenseur 0-D int64.", f"dataset[7] : attendu le label {int(Y_19[7])}, un tenseur 0-D int64 ; reçu {label_19!r}.")
    loader_19 = make_loader_19(data_19, seed=0)
    verdict("20.19", isinstance(loader_19, DataLoader), "make_loader_19 rend un DataLoader.", f"make_loader_19 rend un {type(loader_19).__name__}, attendu un DataLoader.")
    images_b19, labels_b19 = next(iter(make_loader_19(data_19, seed=1)))
    verdict("20.19", tuple(images_b19.shape) == (64, 1, 28, 28) and tuple(labels_b19.shape) == (64,),
            "un mini-batch : des images (64, 1, 28, 28) et des labels (64,) (le DataLoader empile les exemples).",
            f"un mini-batch : attendu (64, 1, 28, 28) et (64,), reçu {tuple(images_b19.shape)} et {tuple(labels_b19.shape)}.")
    epoch1_19 = torch.cat([y for _, y in loader_19])
    epoch2_19 = torch.cat([y for _, y in loader_19])
    again_19 = torch.cat([y for _, y in make_loader_19(data_19, seed=0)])
    verdict("20.19", not torch.equal(epoch1_19, epoch2_19) and sorted(epoch1_19.tolist()) == sorted(Y_19.tolist()),
            "deux epochs : les mêmes 1 000 labels, dans deux ordres différents (shuffle=True).",
            "deux epochs donnent le même ordre, ou pas les 1 000 labels : shuffle=True mélange à chaque epoch.")
    verdict("20.19", torch.equal(epoch1_19, again_19), "le même générateur (même graine) redonne le même ordre : un mélange reproductible.",
            "deux DataLoader créés avec la même graine ne donnent pas le même ordre : passe generator=torch.Generator().manual_seed(seed).")
wb.check("20.19a", n_batches_19)
wb.check("20.19b", last_19)
wb.check("20.19c", dropped_19)

### Ex 20.20 — Zoo des couches : convolution, pooling, récurrente, utilitaires, bruit 📦 ★★ ⏱️ 30 min
**Objectif :** construire et essayer dans PyTorch les couches du tour d'horizon du livre, et suivre les formes.  
**Prérequis :** Ex 20.14 · Ex 20.18 · fiche §20.4.5 à §20.4.8, §20.5 · **Fil rouge :** MNIST · **Parcours :** C

Le tour d'horizon du livre, en PyTorch. Définis :
- `conv_20` : une convolution de 2 filtres 3 × 3 sur un canal, avec un zero-padding de 1 pixel (figure 20.10 du livre : une image de 5 × 4 pixels, `IMAGE_20`) ;
- `max_20` et `avg_20` : le max pooling et le pooling moyen 2 × 2 du petit tenseur `SMALL_20` de 4 × 4 (figure 20.11) ;
- la classe `Noise20(nn.Module)`, une couche de bruit : elle ajoute $\sigma\,\mathcal N(0, 1)$ à son entrée **en entraînement seulement** (`self.training`), et rend son entrée en évaluation ; PyTorch n'en a pas de toute faite ;
- la classe `Zoo20(nn.Module)` : une convolution (1 → 8 canaux, 3 × 3, padding 1), une ReLU, un max pooling 2 × 2, une convolution (8 → 16, 3 × 3, padding 1), une ReLU, un max pooling 2 × 2, ta couche de bruit ($\sigma = 0{,}1$), un aplatissement et une couche dense vers 10 logits ;
- `rnn_20` : une couche `nn.RNN` qui lit les 28 lignes d'une image MNIST comme une séquence de 28 pas de 28 valeurs, avec un état de 32 valeurs (`batch_first=True`).

La vérification applique chaque couche, contrôle les formes et le comportement du bruit (entraînement, évaluation, aucun paramètre), puis affiche `Zoo20` couche par couche, avec la forme de sortie et le nombre de paramètres de chacune.

Dans tes notes : dans le tableau de `Zoo20`, quelles couches ont des paramètres ? Où sont la plupart d'entre eux ? Combien de paramètres a la couche `nn.RNN`, et comment les retrouves-tu (une matrice pour l'entrée, une pour l'état, deux biais) ? Pourquoi la sortie de `rnn_20` a-t-elle une valeur par pas de temps ?

In [ ]:
IMAGE_20 = torch.arange(20.0).reshape(1, 1, 5, 4) / 10      # one image of 5 x 4 pixels, one channel (figure 20.10)
X_20, _ = wb.datasets.load_mnist("train", n=64, normalize=True)
BATCH_20 = torch.from_numpy(X_20).unsqueeze(1)                 # (64, 1, 28, 28)
SMALL_20 = torch.tensor([[[[1.0, 3.0, 2.0, 0.0], [5.0, 4.0, 1.0, 1.0], [0.0, 2.0, 6.0, 8.0], [1.0, 1.0, 3.0, 2.0]]]])   # 4 x 4 (figure 20.11)
print("IMAGE_20", tuple(IMAGE_20.shape), "· BATCH_20", tuple(BATCH_20.shape), "· SMALL_20", tuple(SMALL_20.shape))

In [ ]:
conv_20 = ...      # a convolution with 2 filters of 3 x 3 on 1 channel, with a zero-padding of 1 pixel
max_20 = ...       # SMALL_20 after a max pooling of 2 x 2 (a tensor)
avg_20 = ...       # SMALL_20 after an average pooling of 2 x 2


class Noise20(nn.Module):
    """A noise layer: adds sigma * N(0, 1) to its input during training only; the identity in evaluation."""

    def __init__(self, sigma=0.1):
        super().__init__()
        raise NotImplementedError("Noise20.__init__")

    def forward(self, x):
        raise NotImplementedError("Noise20.forward")


class Zoo20(nn.Module):
    """conv(1 -> 8, 3 x 3, padding 1) -> ReLU -> max pool 2 -> conv(8 -> 16, 3 x 3, padding 1) -> ReLU -> max pool 2
    -> Noise20(0.1) -> flatten -> nn.Linear(16 * 7 * 7, 10): the logits of 10 classes."""

    def __init__(self):
        super().__init__()
        raise NotImplementedError("Zoo20.__init__")

    def forward(self, x):
        raise NotImplementedError("Zoo20.forward")


rnn_20 = ...       # a recurrent layer nn.RNN reading the 28 rows of an image as a sequence of 28 steps of 28 values,
                   # with a state of 32 values (batch_first=True)

In [ ]:
with checked("20.20"):
    if not filled(conv_20):
        print("⏳ Ex 20.20 : définis conv_20.")
    elif not isinstance(conv_20, nn.Conv2d):
        print(f"❌ Ex 20.20 : conv_20 doit être une couche nn.Conv2d, reçu {type(conv_20).__name__}.")
    else:
        out_20 = conv_20(IMAGE_20)
        verdict("20.20", tuple(out_20.shape) == (1, 2, 5, 4), "la convolution de la figure 20.10 : (1, 1, 5, 4) -> (1, 2, 5, 4), une image par filtre, de la taille de l'entrée.",
                f"conv_20(IMAGE_20) a la forme {tuple(out_20.shape)} : attendu (1, 2, 5, 4), 2 filtres et un padding de 1 qui garde 5 x 4.")
        print(f"conv_20 : {n_params(conv_20)} parameters (weight {tuple(conv_20.weight.shape)}, bias {tuple(conv_20.bias.shape)})")
with checked("20.20"):
    if filled(max_20, avg_20):
        same("20.20", "le max pooling 2 x 2 de SMALL_20", max_20, nn.functional.max_pool2d(SMALL_20, 2))
        same("20.20", "le pooling moyen 2 x 2 de SMALL_20", avg_20, nn.functional.avg_pool2d(SMALL_20, 2))
    else:
        print("⏳ Ex 20.20 : définis max_20 et avg_20.")
with checked("20.20"):
    noise_20 = Noise20(sigma=0.5)
    ones_20 = torch.ones(2000, 10)
    noise_20.train()
    noisy_20 = noise_20(ones_20)
    spread_20 = (noisy_20 - ones_20).std().item()
    verdict("20.20", abs(spread_20 - 0.5) < 0.03, f"Noise20(0.5) en entraînement : un bruit d'écart-type {spread_20:.3f}.",
            f"Noise20(0.5) en entraînement : un bruit d'écart-type {spread_20:.3f}, attendu 0,5 (sigma * torch.randn_like(x)).")
    noise_20.eval()
    verdict("20.20", torch.equal(noise_20(ones_20), ones_20), "en évaluation : l'identité.",
            "en évaluation, Noise20 doit rendre son entrée inchangée (teste self.training).")
    verdict("20.20", n_params(noise_20) == 0, "Noise20 n'a aucun paramètre.", f"Noise20 a {n_params(noise_20)} paramètres : une couche de bruit n'apprend rien.")
with checked("20.20"):
    torch.manual_seed(20)
    zoo_20 = Zoo20()
    logits_20 = zoo_20(BATCH_20)
    verdict("20.20", tuple(logits_20.shape) == (64, 10), "Zoo20 rend des logits (64, 10).",
            f"Zoo20 rend la forme {tuple(logits_20.shape)} : attendu (64, 10) (28 -> 14 -> 7 après les deux poolings).")
    print("Zoo20, layer by layer:")
    x_20 = BATCH_20
    for module_20 in [m for m in zoo_20.modules() if not list(m.children())]:
        x_20 = module_20(x_20)
        print(f"   {type(module_20).__name__:10} -> {str(tuple(x_20.shape)):18} {n_params(module_20):6} parameters")
    print("total:", n_params(zoo_20), "parameters")
with checked("20.20"):
    if not filled(rnn_20):
        print("⏳ Ex 20.20 : définis rnn_20.")
    elif not isinstance(rnn_20, nn.RNN):
        print(f"❌ Ex 20.20 : rnn_20 doit être une couche nn.RNN, reçu {type(rnn_20).__name__}.")
    else:
        seq_20, state_20 = rnn_20(BATCH_20[:, 0])          # (64, 28, 28): 64 sequences of 28 rows of 28 values
        verdict("20.20", tuple(seq_20.shape) == (64, 28, 32) and tuple(state_20.shape) == (1, 64, 32),
                "nn.RNN : une sortie (64, 28, 32), un état à chaque pas, et l'état final (1, 64, 32).",
                f"nn.RNN rend {tuple(seq_20.shape)} et {tuple(state_20.shape)} : attendu (64, 28, 32) et (1, 64, 32) (input_size=28, "
                "hidden_size=32, batch_first=True).")
        print(f"rnn_20: {n_params(rnn_20)} parameters")

### Ex 20.21 — Reproduire la figure 20.17 : le plan de VGG16 couche par couche 🎨 ★★ ⏱️ 30 min
**Objectif :** reconstruire VGG16 couche par couche, suivre les formes, et dessiner son plan comme la figure 20.17 du livre.  
**Prérequis :** Ex 20.18 · ✏️ 20.6 · fiche §20.5, §20.6 · **Parcours :** C

Écris :
- `make_vgg16_21()` : VGG16 pour des images de 3 × 224 × 224, un `nn.Sequential` de ses 13 convolutions 3 × 3 (padding 1, chacune suivie d'une ReLU) en 5 blocs, chaque bloc fini par un max pooling 2 × 2, puis un aplatissement et ses 3 couches denses (avec ReLU et dropout de 0,5 entre elles) : la docstring donne la liste ;
- `plan_21(model, x)` : la liste des `(nom de la couche, forme de sortie)`, couche par couche, pour l'entrée `x` (le nom : `type(layer).__name__`) ;
- `draw_plan_21(plan)` : le plan dessiné comme la figure 20.17 : une boîte par couche, de gauche à droite, une couleur par sorte de couche, et la taille de sortie écrite dans la boîte ; elle rend la `Figure` matplotlib.

Puis, avec ton code :
a) `block3_21` : la forme de sortie du 3ᵉ max pooling, en chaîne (comme `"(1, 2, 3)"`) ;
b) `share_21` : la part des paramètres de VGG16 dans sa **première** couche dense, en % (1 décimale).

La vérification compare ton nombre de paramètres à celui de `torchvision.models.vgg16` (construit sans ses poids), contrôle les couches, le plan et la figure, et mesure la durée d'une passe avant sur une image (de quelques dixièmes de seconde sur CPU : VGG16 fait environ 15 milliards de multiplications-additions par image). VGG16 occupe 553 Mo de mémoire : la cellule le libère à la fin.

Dans tes notes : sur ta figure, où la taille des images diminue-t-elle, et où le nombre de canaux augmente-t-il ? Comment le livre et ta figure représentent-ils l'aplatissement ? Qu'est-ce que la figure ne montre pas (pense aux paramètres de chaque couche) ?

In [ ]:
def make_vgg16_21():
    """VGG16 for images of 3 x 224 x 224: an nn.Sequential of its 13 convolutions (3 x 3, padding 1, each followed by a
    ReLU) in 5 blocks of 64, 64 | 128, 128 | 256, 256, 256 | 512, 512, 512 | 512, 512, 512 filters, each block ending
    with nn.MaxPool2d(2); then nn.Flatten(), nn.Linear(25088, 4096), ReLU, nn.Dropout(0.5), nn.Linear(4096, 4096), ReLU,
    nn.Dropout(0.5), nn.Linear(4096, 1000)."""
    raise NotImplementedError("make_vgg16_21")


def plan_21(model, x):
    """The plan of the network: the list of (layer name, output shape as a tuple) for each layer of model (an
    nn.Sequential), fed with x, in order; for example ("Conv2d", (1, 64, 224, 224))."""
    raise NotImplementedError("plan_21")


def draw_plan_21(plan):
    """Draw the plan as in figure 20.17 of the book: one box per layer, from left to right, with one colour per kind of
    layer (convolution, ReLU, pooling, flatten, dense, dropout) and the output size written in the box. Returns the
    matplotlib Figure."""
    raise NotImplementedError("draw_plan_21")


block3_21 = ...    # a) the output shape after the 3rd max pooling, as a string
share_21 = ...     # b) the share of the parameters of VGG16 in its FIRST dense layer, in %

In [ ]:
import torchvision

with torch.device("meta"):                                   # the architecture of torchvision, without allocating its weights
    reference_21 = torchvision.models.vgg16(weights=None)
with checked("20.21"):
    torch.manual_seed(21)
    vgg_21 = make_vgg16_21()
    verdict("20.21", isinstance(vgg_21, nn.Sequential), "make_vgg16_21 rend un nn.Sequential.",
            f"make_vgg16_21 rend un {type(vgg_21).__name__} : attendu un nn.Sequential.")
    count_21, want_21 = f"{n_params(vgg_21):_}".replace("_", " "), f"{n_params(reference_21):_}".replace("_", " ")
    verdict("20.21", n_params(vgg_21) == n_params(reference_21), f"{count_21} paramètres, comme le VGG16 de torchvision.",
            f"{count_21} paramètres, mais torchvision.models.vgg16 en a {want_21} : compare tes blocs et tes couches denses à la docstring.")
    kinds_21 = [type(m).__name__ for m in vgg_21]
    verdict("20.21", kinds_21.count("Conv2d") == 13 and kinds_21.count("Linear") == 3 and kinds_21.count("MaxPool2d") == 5,
            "13 convolutions, 5 max poolings et 3 couches denses.",
            f"{kinds_21.count('Conv2d')} convolutions, {kinds_21.count('MaxPool2d')} max poolings et {kinds_21.count('Linear')} couches denses : attendu 13, 5 et 3.")
    started_21 = time.perf_counter()
    plan_c21 = plan_21(vgg_21, torch.zeros(1, 3, 224, 224))
    print(f"a forward pass of VGG16 on one image: {time.perf_counter() - started_21:.2f} s")
    ok_21 = isinstance(plan_c21, list) and len(plan_c21) == len(vgg_21) and all(isinstance(p, tuple) and len(p) == 2 for p in plan_c21)
    verdict("20.21", ok_21 and plan_c21[-1] == ("Linear", (1, 1000)) and plan_c21[0] == ("Conv2d", (1, 64, 224, 224)),
            "le plan : une ligne par couche, de (Conv2d, (1, 64, 224, 224)) à (Linear, (1, 1000)).",
            f"plan_21 doit rendre une liste de (nom, forme), une par couche ; reçu {str(plan_c21)[:200]}.")
    if ok_21:
        fig_21 = draw_plan_21(plan_c21)
        boxes_21 = [p for ax in getattr(fig_21, "axes", []) for p in ax.patches]
        verdict("20.21", hasattr(fig_21, "axes") and len(boxes_21) >= len(plan_c21),
                f"draw_plan_21 dessine {len(boxes_21)} formes pour {len(plan_c21)} couches.",
                "draw_plan_21 doit rendre une figure matplotlib avec au moins une forme (un rectangle) par couche.")
        plt.show()
    del vgg_21
wb.check("20.21a", shape_text(block3_21))
wb.check("20.21b", share_21)

### Ex 20.22 — batchnorm_forward et batchnorm_backward 🔨 ★★★ ⏱️ 75 min
**Objectif :** écrire la batchnorm de PyTorch en NumPy, entraînement et évaluation, passe avant et passe arrière, en 2-D et en 4-D.  
**Prérequis :** ch. 18 (passe arrière d'une couche) · ✏️ 20.4 · ∂ 20.8 · fiche §20.4.4 · **Fil rouge :** synthétique · **mylearn :** `nn/regularization.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/nn/regularization.py` (créé par `python tools/start_chapter.py 20`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy et la bibliothèque standard seulement : PyTorch est l'**oracle** des tests (sans PyTorch, ils comparent à une transcription NumPy des formules de la fiche, et aucun n'est ignoré). La cellule de vérification recharge ta librairie, essaie les exemples des docstrings et quelques cas choisis (comparés à PyTorch), puis lance les tests de l'exercice ; `python -m pytest tests/test_ch20_regularization.py -q` les lance tous dans un terminal.

**Contrat commun** (les docstrings, plus ces précisions que vérifient les tests) : tes fonctions acceptent des listes comme des tableaux ; elles ne modifient **jamais** leurs entrées (sauf, dans `batchnorm_forward` en entraînement, `running_mean` et `running_var`, mis à jour **en place**, et les tableaux passés à `EarlyStopping.restore`) ; les tableaux qu'elles rendent sont de **nouveaux** tableaux `float64` (jamais l'entrée elle-même, ni une vue) ; les entrées invalides lèvent `ValueError`, sans diffusion (*broadcasting*) : une forme différente est refusée même si NumPy saurait la diffuser.

Écris `batchnorm_forward(x, gamma, beta, running_mean, running_var, training=True, momentum=0.1, eps=1e-5)` et `batchnorm_backward(dout, cache)`. **Contrat** propre à la batchnorm :
- `x` a la forme `(N, D)` (statistiques sur l'axe 0) ou `(N, C, H, W)` (statistiques par canal, sur les axes 0, 2 et 3) ; `gamma`, `beta`, `running_mean` et `running_var` ont la forme `(D,)` ou `(C,)`, sinon `ValueError` (même pour une forme diffusable) ; un autre nombre de dimensions lève aussi `ValueError` ;
- en entraînement : la moyenne et la variance **biaisée** du mini-batch normalisent ($\epsilon$ sous la racine), puis $y = \gamma\hat x + \beta$ ; `running_mean` et `running_var` sont mis à jour **en place** : $(1 - \rho)\times$ l'ancienne valeur $+ \rho\times$ la nouvelle, avec $\rho$ = `momentum` et la variance **non biaisée** pour `running_var` ; une seule valeur par feature (ou par canal) lève `ValueError` ;
- en évaluation : les statistiques glissantes normalisent, et ne changent pas ;
- `cache` est ce que tu veux (un dictionnaire) : tout ce dont la passe arrière a besoin ; `batchnorm_backward` rend `(dx, dgamma, dbeta)`, avec la formule de ∂ 20.8 en entraînement, et $\gamma\,g/\sqrt{\sigma^{2,\text{run}} + \epsilon}$ en évaluation.

La vérification essaie l'exemple de la docstring, puis, en `(6, 3)` et en `(4, 3, 5, 5)`, compare ta passe avant, tes statistiques glissantes, tes gradients et ton évaluation à `nn.BatchNorm1d` et `nn.BatchNorm2d` (en `float64`, avec autograd), essaie deux refus, puis lance les tests (`-k "test_batchnorm_"`).

Dans tes notes : écris l'équation de ∂ 20.8 que tu as programmée, et dis combien d'opérations elle coûte. Pourquoi PyTorch garde-t-il dans `running_var` la variance non biaisée, et pas celle qui sert à normaliser ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="20")   # reload your saved file
with checked("20.22"):
    R = mylearn.nn.regularization
    x_22 = np.array([[1.0, 2.0], [3.0, 6.0]])
    running_mean_22, running_var_22 = np.zeros(2), np.ones(2)
    out_22, cache_22 = pair("20.22", "batchnorm_forward", R.batchnorm_forward(x_22, np.ones(2), np.zeros(2), running_mean_22, running_var_22),
                            ("out", "cache"))
    same("20.22", "l'exemple de la docstring", out_22, [[-0.999995, -0.99999875], [0.999995, 0.99999875]], rtol=1e-6, atol=1e-8)
    same("20.22", "running_mean et running_var après ce pas, mis à jour en place", np.stack([running_mean_22, running_var_22]), [[0.2, 0.4], [1.1, 1.7]])
    rng_22 = np.random.default_rng(22)
    for shape_22 in [(6, 3), (4, 3, 5, 5)]:
        x_r22 = rng_22.normal(2.0, 3.0, size=shape_22)
        gamma_22, beta_22 = rng_22.normal(1.0, 0.2, 3), rng_22.normal(0.0, 0.5, 3)
        layer_22 = (nn.BatchNorm1d if len(shape_22) == 2 else nn.BatchNorm2d)(3, dtype=torch.float64)
        with torch.no_grad():
            layer_22.weight.copy_(torch.tensor(gamma_22))
            layer_22.bias.copy_(torch.tensor(beta_22))
        x_t22 = torch.tensor(x_r22, requires_grad=True)
        want_22 = layer_22.train()(x_t22)
        rm_22, rv_22 = np.zeros(3), np.ones(3)
        got_22, cache_r22 = pair("20.22", "batchnorm_forward", R.batchnorm_forward(x_r22, gamma_22, beta_22, rm_22, rv_22), ("out", "cache"))
        same("20.22", f"entraînement, x de forme {shape_22}, contre nn.{type(layer_22).__name__}", got_22, want_22)
        same("20.22", f"statistiques glissantes après ce pas, forme {shape_22}", np.stack([rm_22, rv_22]),
             torch.stack([layer_22.running_mean, layer_22.running_var]))
        dout_22 = rng_22.normal(size=shape_22)
        want_22.backward(torch.tensor(dout_22))
        dx_22, dgamma_22, dbeta_22 = pair("20.22", "batchnorm_backward", R.batchnorm_backward(dout_22, cache_r22), ("dx", "dgamma", "dbeta"))
        same("20.22", f"dx contre l'autograd de PyTorch, forme {shape_22}", dx_22, x_t22.grad)
        same("20.22", f"dgamma et dbeta contre l'autograd, forme {shape_22}", np.stack([as_numpy(dgamma_22), as_numpy(dbeta_22)]),
             torch.stack([layer_22.weight.grad, layer_22.bias.grad]))
        layer_22.eval()
        got_e22, _ = pair("20.22", "batchnorm_forward", R.batchnorm_forward(x_r22, gamma_22, beta_22, rm_22.copy(), rv_22.copy(), training=False),
                          ("out", "cache"))
        same("20.22", f"évaluation, forme {shape_22} (statistiques glissantes)", got_e22, layer_22(torch.tensor(x_r22)))
    refuses("20.22", "un seul exemple en entraînement", lambda: R.batchnorm_forward(np.ones((1, 3)), np.ones(3), np.zeros(3), np.zeros(3), np.ones(3)))
    refuses("20.22", "gamma de forme (1,) pour 3 features", lambda: R.batchnorm_forward(np.ones((4, 3)), np.ones(1), np.zeros(3), np.zeros(3), np.ones(3)))
if written(lambda: mylearn.nn.regularization.batchnorm_forward(np.ones((2, 1)), np.ones(1), np.zeros(1), np.zeros(1), np.ones(1))):
    run_reg_tests("test_batchnorm_")
else:
    print("⏳ Ex 20.22 : les tests attendent tes fonctions.")

## Partie C · La boucle d'entraînement, ses bugs, ses courbes, et ce qu'on en garde

Écrire la boucle d'entraînement complète et l'évaluation (20.23), réparer une boucle qui contient cinq erreurs classiques (20.24), lire des courbes d'entraînement et de validation (20.25), confronter un modèle ImageNet moderne à des images qu'il n'a jamais vues (20.26), puis sauvegarder un entraînement et le reprendre à l'identique (20.27).

### Ex 20.23 — La boucle d'entraînement PyTorch complète sur MNIST 📦 ★★★ ⏱️ 45 min
**Objectif :** écrire une epoch d'entraînement et une évaluation PyTorch, avec les modes train et eval et le device, et entraîner un MLP sur MNIST.  
**Prérequis :** Ex 20.18 · Ex 20.19 · ch. 19 (optimiseurs) · fiche §20.3.2, §20.7, section PyTorch en pratique (la boucle) · **Fil rouge :** MNIST · **Parcours :** R, C

Écris les deux fonctions de toute boucle d'entraînement :
- `train_one_epoch_23(model, loader, loss_fn, optimizer, device)` : met le modèle en mode entraînement, puis, pour chaque mini-batch de `loader`, envoie `X` et `y` sur `device`, fait **une** passe avant, calcule la loss, puis `optimizer.zero_grad()`, `loss.backward()` et `optimizer.step()` ; elle rend la loss moyenne de l'epoch **par exemple** (chaque loss de mini-batch pondérée par son nombre d'exemples), en `float` Python ;
- `evaluate_23(model, loader, loss_fn, device)` : met le modèle en mode évaluation et calcule sous `torch.no_grad()` ; elle rend le couple `(loss moyenne, accuracy)`, deux `float` Python (moyennes par exemple), sans changer les paramètres.

La vérification lance d'abord un **contrôle** reproductible (le modèle `make_model_23(0)`, fourni, avec dropout ; une graine fixée pour le dropout ; SGD de learning rate 0,1 ; une epoch sur les 2 048 premières images) : elle regarde, pendant chaque passe avant, le mode du modèle et si les gradients sont calculés, puis vérifie :
a) la loss d'entraînement de cette epoch ;
b) l'accuracy de validation qui suit (les 10 000 dernières images d'entraînement, jamais vues).
Puis elle entraîne le modèle avec Adam (learning rate $10^{-3}$) pendant 3 epochs (10 en mode complet) et affiche les courbes en chiffres.

Dans tes notes : pourquoi la loss d'entraînement de la première epoch est-elle plus haute que la loss de validation mesurée juste après ? (Deux raisons.) Si la loss de validation cessait de baisser, quel planning du ch. 19 pourrais-tu ajouter, et où l'appellerais-tu (`torch.optim.lr_scheduler.ReduceLROnPlateau`) ?

In [ ]:
X_ALL_23, Y_ALL_23 = wb.datasets.load_mnist("train", normalize=True)        # 60 000 images, float32 in [0, 1]
N_TRAIN_23 = wb.by_mode(fast=12_000, full=50_000)                            # the training images used here
TRAIN_SET_23 = TensorDataset(torch.from_numpy(X_ALL_23[:N_TRAIN_23]).unsqueeze(1), torch.from_numpy(Y_ALL_23[:N_TRAIN_23]))
VAL_SET_23 = TensorDataset(torch.from_numpy(X_ALL_23[50_000:]).unsqueeze(1), torch.from_numpy(Y_ALL_23[50_000:]))
VAL_LOADER_23 = DataLoader(VAL_SET_23, batch_size=1000)                      # the last 10 000 images: never trained on


def train_loader_23(seed, n=None):
    """The training mini-batches (64 images), shuffled by a generator seeded with seed; the first n images if n is given."""
    data = TRAIN_SET_23 if n is None else torch.utils.data.Subset(TRAIN_SET_23, range(n))
    return DataLoader(data, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(seed))


def make_model_23(seed=0):
    """The MLP of this part, 784-128-10 with a dropout of 0.2, initialized with the seed, on the device DEVICE."""
    torch.manual_seed(seed)
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 10)).to(DEVICE)


print(f"training: {len(TRAIN_SET_23)} images · validation: {len(VAL_SET_23)} images · device {DEVICE}")

In [ ]:
def train_one_epoch_23(model, loader, loss_fn, optimizer, device):
    """One epoch of training of model (already on device) over the mini-batches of loader.

    Puts the model in training mode, then, for each mini-batch: moves X and y to device, ONE forward pass, the loss,
    optimizer.zero_grad(), loss.backward(), optimizer.step(). Returns the mean training loss over the EXAMPLES of the
    epoch (each mini-batch loss weighted by its number of examples), as a Python float.
    """
    raise NotImplementedError("train_one_epoch_23")


def evaluate_23(model, loader, loss_fn, device):
    """(mean loss, accuracy) of model on the mini-batches of loader, as two Python floats (means over the examples).

    Puts the model in evaluation mode and computes under torch.no_grad(); the parameters do not change.
    """
    raise NotImplementedError("evaluate_23")

In [ ]:
control_loss_23 = control_acc_23 = ...
with checked("20.23"):
    model_23 = make_model_23(0)
    seen_23 = []                                              # (training mode, gradients enabled) at every forward pass
    hook_23 = model_23.register_forward_pre_hook(lambda module, args: seen_23.append((module.training, torch.is_grad_enabled())))
    try:
        torch.manual_seed(1)                                  # the dropout masks of the control run
        control_loss_23 = train_one_epoch_23(model_23, train_loader_23(23, n=2048), nn.CrossEntropyLoss(),
                                             torch.optim.SGD(model_23.parameters(), lr=0.1), DEVICE)
        train_seen_23, seen_23[:] = list(seen_23), []
        before_23 = [p.detach().clone() for p in model_23.parameters()]
        first_23 = evaluate_23(model_23, VAL_LOADER_23, nn.CrossEntropyLoss(), DEVICE)
        eval_seen_23 = list(seen_23)
        second_23 = evaluate_23(model_23, VAL_LOADER_23, nn.CrossEntropyLoss(), DEVICE)
    finally:
        hook_23.remove()
    verdict("20.23", isinstance(control_loss_23, float), "train_one_epoch_23 rend un float Python.",
            f"train_one_epoch_23 rend {type(control_loss_23).__name__} : attendu un float Python (additionne loss.item(), pas loss : "
            "un tenseur garde tout le graphe de calcul).")
    verdict("20.23", len(train_seen_23) == 32 and all(mode and grad for mode, grad in train_seen_23),
            "entraînement : 32 passes avant (une par mini-batch), en mode train, avec les gradients.",
            f"entraînement : {len(train_seen_23)} passes avant (attendu 32, UNE par mini-batch de 2 048 / 64), modes "
            f"{sorted(set(train_seen_23))} (attendu (True, True) : model.train() au début, et pas de torch.no_grad()).")
    ok_eval_23 = isinstance(first_23, tuple) and len(first_23) == 2
    if not ok_eval_23:
        print(f"❌ Ex 20.23 : evaluate_23 doit rendre le couple (loss, accuracy), reçu {first_23!r}.")
    else:
        verdict("20.23", bool(eval_seen_23) and all(not mode and not grad for mode, grad in eval_seen_23),
                "évaluation : mode eval et torch.no_grad() à chaque passe avant.",
                f"évaluation : modes (train, gradients) vus {sorted(set(eval_seen_23))} : attendu (False, False), avec model.eval() "
                "et torch.no_grad().")
        unchanged_23 = all(torch.equal(a, b) for a, b in zip(before_23, model_23.parameters()))
        verdict("20.23", unchanged_23 and first_23 == second_23 and all(isinstance(v, float) for v in first_23),
                "evaluate_23 ne change pas les paramètres, rend deux floats et redonne les mêmes valeurs.",
                f"evaluate_23 : deux appels donnent {first_23} puis {second_23}, ou changent les paramètres : attendu deux floats, "
                "identiques d'un appel à l'autre (le dropout doit être éteint).")
        control_acc_23 = first_23[1]
        print(f"control run: training loss {float(control_loss_23):.4f}, validation loss {first_23[0]:.4f}, accuracy {first_23[1]:.4f}")
with checked("20.23"):                                        # the mean over the EXAMPLES: 100 images, mini-batches of 64 and 36
    if isinstance(control_loss_23, float):
        with torch.random.fork_rng():                         # leaves the random state of the next trainings untouched
            torch.manual_seed(0)
            probe_23 = nn.Sequential(nn.Flatten(), nn.Linear(784, 10)).to(DEVICE)
            got_23 = train_one_epoch_23(probe_23, DataLoader(torch.utils.data.Subset(TRAIN_SET_23, range(100)), batch_size=64),
                                        nn.CrossEntropyLoss(), torch.optim.SGD(probe_23.parameters(), lr=0.0), DEVICE)
            with torch.no_grad():
                want_23 = nn.functional.cross_entropy(probe_23(TRAIN_SET_23.tensors[0][:100].to(DEVICE)),
                                                      TRAIN_SET_23.tensors[1][:100].to(DEVICE)).item()
        verdict("20.23", isinstance(got_23, float) and abs(got_23 - want_23) < 1e-5,
                "la loss rendue est la moyenne sur les exemples (sur 100 images, le mini-batch de 36 pèse 36, pas autant que celui de 64).",
                f"sur 100 images en mini-batches de 64 et 36 (learning rate nul), train_one_epoch_23 rend {got_23!r}, la moyenne "
                f"sur les exemples vaut {want_23:.5f} : pondère chaque loss de mini-batch par son nombre d'exemples.")
if control_loss_23 is ... or isinstance(control_loss_23, float):
    wb.check("20.23a", control_loss_23, computed=True)
else:
    print("❌ Ex 20.23a : la loss du contrôle n'est vérifiée que si train_one_epoch_23 rend un float Python (voir plus haut).")
wb.check("20.23b", control_acc_23, computed=True)
with checked("20.23"):
    if isinstance(control_loss_23, float) and isinstance(control_acc_23, float):
        model_23 = make_model_23(0)
        optimizer_23 = torch.optim.Adam(model_23.parameters(), lr=1e-3)
        print("epoch   train loss   val loss   val accuracy   seconds")
        for epoch_23 in range(1, wb.by_mode(fast=3, full=10) + 1):
            started_23 = time.perf_counter()
            train_l_23 = train_one_epoch_23(model_23, train_loader_23(epoch_23), nn.CrossEntropyLoss(), optimizer_23, DEVICE)
            val_l_23, val_a_23 = evaluate_23(model_23, VAL_LOADER_23, nn.CrossEntropyLoss(), DEVICE)
            print(f"{epoch_23:5}   {train_l_23:10.4f}   {val_l_23:8.4f}   {val_a_23:12.4f}   {time.perf_counter() - started_23:7.1f}")
        verdict("20.23", val_a_23 > 0.92, f"après l'entraînement, {val_a_23:.1%} de bonnes réponses en validation.",
                f"après l'entraînement, {val_a_23:.1%} en validation : attendu plus de 92 % ; vérifie la boucle.")
    else:
        print("⏳ Ex 20.23 : l'entraînement complet attend le contrôle ci-dessus.")

### Ex 20.24 — Chasse aux bugs PyTorch : cinq erreurs classiques 🐛 ★★ ⏱️ 30 min
**Objectif :** trouver et corriger cinq erreurs classiques d'une boucle PyTorch, chacune à son symptôme.  
**Prérequis :** Ex 20.23 · fiche, section PyTorch en pratique et tableau des pièges · **Fil rouge :** MNIST · **Parcours :** R, C

Un collègue a écrit ce code pour entraîner un MLP sur MNIST : la classe `Net24`, une epoch d'entraînement `train_epoch_24` et une évaluation `evaluate_24`. Il contient **cinq** erreurs classiques. Corrige-les dans la cellule (garde les noms et les signatures), sans réécrire ce qui est juste.

La vérification les cherche dans l'ordre où elles se masquent, une par point : 1, la forme de ce que rend le réseau ; 2, ce que rend sa dernière couche ; 3, le gradient utilisé à chaque pas (sur un petit modèle) ; 4, le type de ce que rend `train_epoch_24` ; 5, l'évaluation d'un modèle qui a du dropout. Quand les cinq passent, elle entraîne `Net24` une epoch et mesure son accuracy.

Dans tes notes : pour chaque erreur, le symptôme qu'elle aurait produit dans un vrai entraînement (un plantage, une loss qui stagne, une mémoire qui grossit, une accuracy qui varie…), et la ligne que tu as changée.

In [ ]:
X_24, Y_24 = wb.datasets.load_mnist("train", n=6000, normalize=True)
LOADER_24 = DataLoader(TensorDataset(torch.from_numpy(X_24).unsqueeze(1), torch.from_numpy(Y_24)), batch_size=64, shuffle=True,
                       generator=torch.Generator().manual_seed(24))
print("6 000 MNIST images in mini-batches of 64")

In [ ]:
class Net24(nn.Module):
    """A MLP 784-128-10 with a dropout of 0.3; it returns the logits."""

    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(784, 128)
        self.drop = nn.Dropout(0.3)
        self.out = nn.Linear(128, 10)

    def forward(self, x):
        x = torch.flatten(x)
        x = self.drop(torch.relu(self.hidden(x)))
        return torch.softmax(self.out(x), dim=1)


def train_epoch_24(model, loader, optimizer):
    """One epoch of training (cross-entropy on the logits); returns the mean loss of the epoch, a Python float."""
    model.train()
    total, count = 0.0, 0
    for X, y in loader:
        loss = nn.functional.cross_entropy(model(X), y)
        loss.backward()
        optimizer.step()
        total += loss * len(y)
        count += len(y)
    return total / count


def evaluate_24(model, loader):
    """The accuracy of model on the mini-batches of loader, a Python float."""
    correct, count = 0, 0
    for X, y in loader:
        correct += (model(X).argmax(dim=1) == y).sum().item()
        count += len(y)
    return correct / count

In [ ]:
points_24 = {}
with checked("20.24"):                                    # point 1: the shape of the logits
    torch.manual_seed(0)
    try:
        logits_24 = Net24()(torch.rand(8, 1, 28, 28))
    except RuntimeError as exc:
        logits_24 = None
        print(f"❌ Ex 20.24, point 1 : Net24 plante sur un mini-batch de 8 images : {str(exc)[:160]}")
    points_24[1] = logits_24 is not None and tuple(logits_24.shape) == (8, 10)
    if logits_24 is not None:
        verdict("20.24", points_24[1], "point 1 : Net24 rend une ligne de 10 scores par image.",
                f"point 1 : Net24 rend la forme {tuple(logits_24.shape)} pour 8 images : attendu (8, 10), une ligne par image.")
with checked("20.24"):                                    # point 2: logits, not probabilities
    if points_24.get(1):
        rows_24 = logits_24.detach().sum(dim=1)
        points_24[2] = not (torch.allclose(rows_24, torch.ones(8), atol=1e-5) and bool((logits_24 >= 0).all()))
        verdict("20.24", points_24[2], "point 2 : Net24 rend des logits.",
                "point 2 : les sorties de Net24 sont positives et chaque ligne somme à 1 : regarde ce que la loss attend.")
    else:
        print("⏳ Ex 20.24, point 2 : répare d'abord le point 1.")
with checked("20.24"):                                    # points 3 and 4: one epoch on a small probe
    torch.manual_seed(1)
    probe_24 = nn.Linear(4, 3)
    batches_24 = [(torch.randn(5, 4), torch.randint(0, 3, (5,))) for _ in range(3)]
    returned_24 = train_epoch_24(probe_24, batches_24, torch.optim.SGD(probe_24.parameters(), lr=0.0))   # lr 0: no move
    last_24 = nn.Linear(4, 3)
    last_24.load_state_dict(probe_24.state_dict())
    nn.functional.cross_entropy(last_24(batches_24[-1][0]), batches_24[-1][1]).backward()   # the gradient of the last batch alone
    grad_24 = probe_24.weight.grad
    alone_24 = grad_24 is None or bool((grad_24 == 0).all()) or torch.allclose(grad_24, last_24.weight.grad, atol=1e-7)
    torch.manual_seed(1)
    mover_24 = nn.Linear(4, 3)
    start_24 = mover_24.weight.detach().clone()
    train_epoch_24(mover_24, batches_24, torch.optim.SGD(mover_24.parameters(), lr=0.1))
    moved_24 = not torch.equal(mover_24.weight.detach(), start_24)
    points_24[3] = alone_24 and moved_24
    if not alone_24:
        print("❌ Ex 20.24, point 3 : après trois mini-batches, le gradient vaut la SOMME des trois gradients : regarde ce que fait backward() à .grad.")
    elif not moved_24:
        print("❌ Ex 20.24, point 3 : avec un learning rate de 0,1, les poids n'ont pas bougé : les gradients sont-ils effacés avant step() ?")
    else:
        print("✅ Ex 20.24 : point 3 : chaque pas utilise le gradient de son seul mini-batch.")
    points_24[4] = isinstance(returned_24, float)
    verdict("20.24", points_24[4], "point 4 : train_epoch_24 rend un float Python.",
            f"point 4 : train_epoch_24 rend {type(returned_24).__name__}" + (" avec un grad_fn" if getattr(returned_24, "grad_fn", None) is not None else "")
            + " : attendu un float" + (" (et la somme garde ici tout le graphe de calcul de l'epoch en mémoire)." if getattr(returned_24, "grad_fn", None) is not None else "."))
with checked("20.24"):                                    # point 5: the evaluation
    torch.manual_seed(2)
    noisy_24 = nn.Sequential(nn.Linear(4, 64), nn.Dropout(0.5), nn.Linear(64, 3))
    probe_x24 = [(torch.randn(200, 4), torch.randint(0, 3, (200,)))]
    modes_24 = []
    hook_24 = noisy_24.register_forward_pre_hook(lambda module, args: modes_24.append((module.training, torch.is_grad_enabled())))
    try:
        accs_24 = [evaluate_24(noisy_24, probe_x24) for _ in range(3)]
    finally:
        hook_24.remove()
    points_24[5] = len(set(accs_24)) == 1 and all(not mode for mode, _ in modes_24)
    verdict("20.24", points_24[5], "point 5 : l'évaluation est déterministe (pas de dropout).",
            f"point 5 : trois évaluations du même modèle donnent {accs_24} : regarde ce que fait le dropout pendant la mesure.")
    if points_24[5] and any(grad for _, grad in modes_24):
        print("ℹ️ Ex 20.24, point 5 : l'évaluation calcule encore le graphe des gradients (torch.no_grad() le désactive, et économise la mémoire).")
with checked("20.24"):
    if all(points_24.get(k) for k in range(1, 6)):
        torch.manual_seed(24)
        net_24 = Net24()
        loss_24 = train_epoch_24(net_24, LOADER_24, torch.optim.Adam(net_24.parameters(), lr=1e-3))
        acc_24 = evaluate_24(net_24, LOADER_24)
        verdict("20.24", acc_24 > 0.85, f"tout est réparé : une epoch, loss {loss_24:.3f}, accuracy {acc_24:.1%} (sur les images d'entraînement).",
                f"une epoch donne {acc_24:.1%} : il reste une erreur.")
    else:
        print("⏳ Ex 20.24 : l'entraînement de contrôle attend les cinq points.")

### Ex 20.25 — Courbes train/validation : overfitting, eval() oublié, fuite 📈 ★★ ⏱️ 20 min
**Objectif :** reconnaître sur des courbes d'entraînement et de validation l'overfitting, une fuite, un eval() oublié, l'underfitting et l'effet du dropout.  
**Prérequis :** ch. 9 (overfitting, early stopping) · 🧠 20.Q11 · fiche §20.4.3, sections PyTorch en pratique et régulariser un réseau · **Fil rouge :** Fashion-MNIST · **Parcours :** R, M

Cinq entraînements d'un classifieur sur Fashion-MNIST (10 classes ; les courbes sont fabriquées, mais leurs formes sont celles qu'on rencontre) : à chaque epoch, la loss d'entraînement (la moyenne pendant l'epoch, mesurée **avec** le dropout quand le modèle en a), la loss de validation et l'accuracy de validation (mesurées après l'epoch). Les données sont dans `CURVES_25` si tu veux les lire en chiffres. Réponds par la **lettre** de l'entraînement, sauf b) :
a) `overfit_25` : celui qui surapprend nettement ;
b) `stop_25` : pour cet entraînement, l'epoch dont un early stopping garderait les poids (la première epoch est la n° 1) ;
c) `leak_25` : celui dont la validation contient, sans qu'on le sache, des images d'entraînement (une fuite) ;
d) `noeval_25` : celui dont la validation est mesurée sans `model.eval()` : le dropout reste actif pendant la mesure ;
e) `underfit_25` : celui dont le modèle est bien trop petit pour le problème ;
f) `dropout_25` : celui qui est sain, avec un dropout bien réglé.

Dans tes notes : pour f), pourquoi la loss de validation peut-elle rester **sous** celle d'entraînement sans qu'il y ait de fuite ? Pour c), qu'est-ce qui t'a alerté, et comment vérifierais-tu la fuite sur les données ?

In [ ]:
CURVES_25 = wb.synth.training_curves()       # five training histories: CURVES_25["A"]["val_loss"]...
fig_25, axes_25 = plt.subplots(1, 5, figsize=(20, 3.8), sharey=True)
for ax, (name_25, history_25) in zip(axes_25, CURVES_25.items()):
    epochs_25 = np.arange(1, len(history_25["train_loss"]) + 1)
    ax.plot(epochs_25, history_25["train_loss"], label="loss d'entraînement")
    ax.plot(epochs_25, history_25["val_loss"], label="loss de validation")
    ax.set_title(f"entraînement {name_25}")
    ax.set_xlabel("epoch")
    twin_25 = ax.twinx()
    twin_25.plot(epochs_25, history_25["val_acc"], color="0.6", ls=":", label="accuracy de validation")
    twin_25.set_ylim(0.6, 1.0)
axes_25[0].set_ylabel("loss (nats)")
axes_25[0].legend(fontsize=8)
twin_25.legend(fontsize=8, loc="center right")
plt.tight_layout()
plt.show()

In [ ]:
overfit_25 = ...   # a) a letter, e.g. "Z"
stop_25 = ...      # b) an epoch number (the first epoch is 1)
leak_25 = ...      # c) a letter
noeval_25 = ...    # d) a letter
underfit_25 = ...  # e) a letter
dropout_25 = ...   # f) a letter

In [ ]:
for letter_25, answer_25 in zip("abcdef", [overfit_25, stop_25, leak_25, noeval_25, underfit_25, dropout_25]):
    wb.check(f"20.25{letter_25}", answer_25)

### Ex 20.26 — Un modèle ImageNet face à des images qu'il n'a jamais vues 🔮 ★★ ⏱️ 25 min
**Objectif :** prédire ce que fait un petit modèle ImageNet moderne sur des photos de 32 × 32 pixels et sur du bruit, puis le mesurer.  
**Prérequis :** Ex 20.23 · ch. 17 (softmax) · fiche §20.6, §20.8 · **Fil rouge :** CIFAR-10 · **Parcours :** C

Au §20.6, le livre montre VGG16 face à des objets hors de ses 1 000 classes. On refait l'expérience avec un modèle moderne et léger, **MobileNet V3 Small** (2,5 millions de paramètres, 67,7 % top-1 sur ImageNet, poids fournis par torchvision, environ 10 Mo à télécharger une fois), qui reçoit des images redimensionnées par son prétraitement (agrandies à 256 × 256, puis recadrées au centre en 224 × 224) : 100 photos de **chats** de CIFAR-10 (32 × 32 pixels), 200 photos de CIFAR-10 de toutes les classes, et 200 images de **bruit pur** (des pixels tirés au hasard). Pour chaque image, le modèle rend 1 000 probabilités (une softmax) ; on regarde sa classe la plus probable et sa probabilité la plus haute. Prédis, **avant** d'exécuter quoi que ce soit (pour a), d) et e) : (A) moins de 20 % ; (B) de 20 à 40 % ; (C) de 40 à 60 % ; (D) de 60 à 80 % ; (E) plus de 80 %) :
a) `cats_26` : la part des 100 photos de chats que le modèle classe dans l'une des 5 races de chats domestiques d'ImageNet ;
b) `noise_26` : la moyenne, sur les 200 images de bruit, de la probabilité la plus haute : (A) moins de 0,2 ; (B) de 0,2 à 0,4 ; (C) de 0,4 à 0,6 ; (D) de 0,6 à 0,8 ; (E) plus de 0,8 ;
c) `higher_26` : cette moyenne est-elle plus haute sur les 200 photos de CIFAR-10 que sur le bruit ? (`True` ou `False`) ;
d) `same_26` : la part des images de bruit qui reçoivent **la même** classe, la plus fréquente ;
e) `refused_26` : on décide de refuser de répondre quand la probabilité la plus haute est sous 0,5 ; la part des 200 photos de CIFAR-10 refusées.

Puis exécute l'expérience : elle mesure les cinq valeurs, montre six chats et six bruits avec leurs deux classes les plus probables, et vérifie tes prédictions (sans réseau, elle te renvoie vers Colab).

In [ ]:
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

try:                                                       # ≈ 10 MB, downloaded once into the cache of torch
    WEIGHTS_26 = MobileNet_V3_Small_Weights.DEFAULT
    MODEL_26 = mobilenet_v3_small(weights=WEIGHTS_26).eval()
    PREPROCESS_26, CATEGORIES_26 = WEIGHTS_26.transforms(), WEIGHTS_26.meta["categories"]
    print(f"MobileNet V3 Small: {n_params(MODEL_26):_} parameters, {len(CATEGORIES_26)} ImageNet classes".replace("_", " "))
except Exception as exc:                                   # no network: the experiment says so
    MODEL_26 = None
    print(f"ℹ️ Ex 20.26 : impossible de télécharger les poids du modèle ({type(exc).__name__}) : fais cet exercice sur Colab.")
X_C26, Y_C26 = wb.datasets.load_cifar10("test", channels_first=True)          # uint8 (10000, 3, 32, 32)
CATS_26 = X_C26[np.flatnonzero(Y_C26 == 3)[:100]]          # 100 photos of cats (class 3 of CIFAR-10)
PHOTOS_26 = X_C26[:200]                                    # 200 photos of the 10 classes
NOISE_26 = np.random.default_rng(0).integers(0, 256, size=(200, 3, 32, 32), dtype=np.uint8)   # 200 images of pure noise
DOMESTIC_CATS_26 = range(281, 286)                         # the 5 classes of domestic cats among the 1 000 of ImageNet


def probabilities_26(images):
    """The softmax probabilities (N, 1000) of the ImageNet model for uint8 images (N, 3, H, W)."""
    with torch.no_grad():
        return torch.softmax(MODEL_26(PREPROCESS_26(torch.from_numpy(images))), dim=1)

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
cats_26 = ...      # a) "A", "B", "C", "D" or "E"
noise_26 = ...     # b) "A", "B", "C", "D" or "E"
higher_26 = ...    # c) True or False
same_26 = ...      # d) "A", "B", "C", "D" or "E"
refused_26 = ...   # e) "A", "B", "C", "D" or "E"

In [ ]:
if any(answer is ... or answer is None for answer in [cats_26, noise_26, higher_26, same_26, refused_26]):
    print("⏳ Ex 20.26 : écris d'abord tes cinq prédictions, puis relance cette cellule.")
else:
    if MODEL_26 is None:
        print("⏳ Ex 20.26 : pas de modèle (pas de réseau) : fais cette expérience sur Colab.")
    else:
        def band_26(share):
            """The letter of a share: A below 20 %, B from 20 to 40 %, C from 40 to 60 %, D from 60 to 80 %, E above 80 %."""
            return "ABCDE"[min(4, int(share * 5))]

        p_cats_26, p_photos_26, p_noise_26 = probabilities_26(CATS_26), probabilities_26(PHOTOS_26), probabilities_26(NOISE_26)
        share_cats_26 = float(np.isin(p_cats_26.argmax(dim=1).numpy(), list(DOMESTIC_CATS_26)).mean())
        conf_photos_26, conf_noise_26 = p_photos_26.max(dim=1).values, p_noise_26.max(dim=1).values
        top_noise_26 = p_noise_26.argmax(dim=1).numpy()
        values_26, counts_26 = np.unique(top_noise_26, return_counts=True)
        share_same_26 = counts_26.max() / len(top_noise_26)
        share_refused_26 = float((conf_photos_26 < 0.5).float().mean())
        print(f"a) cat photos classified as a domestic cat: {share_cats_26:.0%}")
        print(f"b) mean of the highest probability on noise: {conf_noise_26.mean():.3f}")
        print(f"c) the same on the CIFAR-10 photos: {conf_photos_26.mean():.3f}")
        print(f"d) noise images that get the most frequent class, « {CATEGORIES_26[values_26[counts_26.argmax()]]} »: {share_same_26:.0%}")
        print(f"e) CIFAR-10 photos refused by the rule « highest probability < 0.5 »: {share_refused_26:.0%}")
        fig_26, axes_26 = plt.subplots(2, 6, figsize=(16, 6))
        for ax, image_26, probs_26 in zip(axes_26.ravel(), np.concatenate([CATS_26[:6], NOISE_26[:6]]),
                                          torch.cat([p_cats_26[:6], p_noise_26[:6]])):
            ax.imshow(image_26.transpose(1, 2, 0))
            best_26 = probs_26.topk(2)
            ax.set_title("\n".join(f"{CATEGORIES_26[i][:16]} {p:.2f}" for p, i in zip(best_26.values.tolist(), best_26.indices.tolist())), fontsize=8)
            ax.axis("off")
        plt.tight_layout()
        plt.show()
        for letter_26, value_26 in zip("abcde", [cats_26, noise_26, higher_26, same_26, refused_26]):
            wb.check(f"20.26{letter_26}", value_26)

Dans tes notes : pour chaque prédiction, qu'est-ce qui t'a guidé, et qu'est-ce qui t'a surpris ? Qu'est-ce que l'expérience dit de la règle e) pour détecter qu'une image n'est pas « pour » le modèle ? Que changerais-tu pour reconnaître les chats de CIFAR-10 ?

### Ex 20.27 — Sauvegarder, recharger et reproduire un entraînement 🛠️ ★★ ⏱️ 25 min
**Objectif :** écrire un checkpoint qui permet de reprendre un entraînement exactement là où il s'était arrêté.  
**Prérequis :** Ex 20.23 · fiche, section PyTorch en pratique (sauvegarder) et encadré sur la reproductibilité · **Fil rouge :** MNIST · **Parcours :** R, C

Un entraînement s'arrête (une machine qui redémarre, une session Colab qui expire) : il faut pouvoir le reprendre. Écris :
- `save_checkpoint_27(path, model, optimizer, epoch, generator)` : sauvegarde dans `path` (avec `torch.save`) un dictionnaire de cinq clés : `"model"` (le `state_dict` du modèle), `"optimizer"` (celui de l'optimiseur), `"epoch"` (le nombre d'epochs faites), `"generator"` (l'état du générateur qui mélange les données, `generator.get_state()`) et `"torch_rng"` (l'état du générateur global de torch, qui tire les masques du dropout, `torch.get_rng_state()`) ;
- `load_checkpoint_27(path, model, optimizer, generator)` : recharge tout cela (dans le modèle, l'optimiseur, le générateur et le générateur global) et rend le nombre d'epochs faites ; `torch.load(path)` doit marcher dans son mode sûr par défaut (`weights_only=True`).

La vérification entraîne un MLP deux epochs d'affilée (la référence) ; puis le même MLP une epoch, sauvegarde, crée un **nouveau** modèle, un nouvel optimiseur et un nouveau générateur, recharge le checkpoint et fait la seconde epoch. Elle contrôle le contenu du fichier, chaque élément rechargé, puis compare les poids finaux à ceux de la référence, **au bit près**. Les fichiers vont dans un dossier temporaire, jamais dans le dépôt.

Dans tes notes : enlève une à une les clés `"optimizer"`, `"generator"` et `"torch_rng"` : qu'est-ce qui casse, et pourquoi ? Sur GPU, que faudrait-il de plus pour une reproduction au bit près (encadré de la fiche) ?

In [ ]:
X_27, Y_27 = wb.datasets.load_mnist("train", n=2048, normalize=True)
DATA_27 = TensorDataset(torch.from_numpy(X_27).unsqueeze(1), torch.from_numpy(Y_27))


def make_model_27(seed):
    """A MLP 784-64-10 with dropout, initialized with the seed."""
    torch.manual_seed(seed)
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 10))


def epoch_27(model, optimizer, generator):
    """One epoch of training on DATA_27, in mini-batches of 64 shuffled by generator."""
    model.train()
    for X, y in DataLoader(DATA_27, batch_size=64, shuffle=True, generator=generator):
        loss = nn.functional.cross_entropy(model(X), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


FOLDER_27 = Path(tempfile.mkdtemp())                        # a temporary folder (never in the repository)
print("checkpoints go to", FOLDER_27)

In [ ]:
def save_checkpoint_27(path, model, optimizer, epoch, generator):
    """Save in path everything needed to resume the training exactly: a dict with the keys "model" (the state_dict),
    "optimizer" (its state_dict), "epoch" (the number of epochs done), "generator" (the state of the generator that
    shuffles the data) and "torch_rng" (the state of the global random generator of torch, used by the dropout)."""
    raise NotImplementedError("save_checkpoint_27")


def load_checkpoint_27(path, model, optimizer, generator):
    """Load the checkpoint of path into model, optimizer, generator and the global random generator of torch, and
    return the number of epochs done. torch.load must work with its default safe mode (weights_only=True)."""
    raise NotImplementedError("load_checkpoint_27")

In [ ]:
with checked("20.27"):
    model_a27 = make_model_27(0)                                     # A: two epochs in a row, the reference
    opt_a27, gen_a27 = torch.optim.Adam(model_a27.parameters(), lr=1e-3), torch.Generator().manual_seed(27)
    torch.manual_seed(7)
    for _ in range(2):
        epoch_27(model_a27, opt_a27, gen_a27)
    model_b27 = make_model_27(0)                                     # B: one epoch, then a checkpoint
    opt_b27, gen_b27 = torch.optim.Adam(model_b27.parameters(), lr=1e-3), torch.Generator().manual_seed(27)
    torch.manual_seed(7)
    epoch_27(model_b27, opt_b27, gen_b27)
    path_27 = FOLDER_27 / "checkpoint.pt"
    save_checkpoint_27(path_27, model_b27, opt_b27, 1, gen_b27)
    saved_rng_27, saved_gen_27 = torch.get_rng_state(), gen_b27.get_state()
    try:
        content_27 = torch.load(path_27)                             # the safe mode of torch.load (weights_only=True)
    except Exception as exc:
        content_27 = None
        print(f"❌ Ex 20.27 : torch.load(path) échoue en mode sûr ({type(exc).__name__}) : sauvegarde des state_dict, des tenseurs, "
              "des nombres (pas le modèle ni l'optimiseur eux-mêmes).")
    if content_27 is not None:
        keys_27 = set(content_27) if isinstance(content_27, dict) else set()
        verdict("20.27", keys_27 == {"model", "optimizer", "epoch", "generator", "torch_rng"},
                "le checkpoint contient model, optimizer, epoch, generator et torch_rng.",
                f"le checkpoint contient {sorted(keys_27)} : attendu les clés model, optimizer, epoch, generator et torch_rng.")
        torch.manual_seed(999)                                       # C: a NEW model, optimizer and generator, then the checkpoint
        model_c27 = make_model_27(1)
        opt_c27, gen_c27 = torch.optim.Adam(model_c27.parameters(), lr=1e-3), torch.Generator().manual_seed(0)
        done_27 = load_checkpoint_27(path_27, model_c27, opt_c27, gen_c27)
        verdict("20.27", isinstance(done_27, int) and not isinstance(done_27, bool) and done_27 == 1, "load_checkpoint_27 rend le nombre d'epochs faites : 1.", f"load_checkpoint_27 rend {done_27!r} : attendu 1.")
        same_model_27 = all(torch.equal(a, b) for a, b in zip(model_c27.state_dict().values(), model_b27.state_dict().values()))
        state_c27 = opt_c27.state_dict()["state"]
        same_opt_27 = len(state_c27) == len(opt_b27.state_dict()["state"]) and all(
            torch.equal(state_c27[k]["exp_avg"], v["exp_avg"]) for k, v in opt_b27.state_dict()["state"].items())
        same_gen_27 = torch.equal(gen_c27.get_state(), saved_gen_27)
        same_rng_27 = torch.equal(torch.get_rng_state(), saved_rng_27)
        for ok_27, what_27, fix_27 in [(same_model_27, "les poids du modèle", "model.load_state_dict(...)"),
                                       (same_opt_27, "l'état de l'optimiseur (les moyennes d'Adam)", "optimizer.load_state_dict(...)"),
                                       (same_gen_27, "l'état du générateur qui mélange les données", "generator.set_state(...)"),
                                       (same_rng_27, "l'état du générateur global de torch (le dropout)", "torch.set_rng_state(...)")]:
            verdict("20.27", ok_27, f"après le chargement : {what_27}, comme au moment de la sauvegarde.",
                    f"après le chargement, ce n'est pas l'état de la sauvegarde : {what_27} ({fix_27}).")
        epoch_27(model_c27, opt_c27, gen_c27)                        # the second epoch, resumed
        identical_27 = all(torch.equal(a, b) for a, b in zip(model_c27.parameters(), model_a27.parameters()))
        verdict("20.27", identical_27, "la reprise donne exactement les mêmes poids que deux epochs d'affilée (au bit près).",
                "la reprise ne donne pas les mêmes poids que deux epochs d'affilée : quelque chose manque au checkpoint (voir plus haut).")

## Partie D · Régulariser sur Fashion-MNIST, un réseau de 30 couches, puis le défi

Fashion-MNIST (des vêtements, plus difficile que les chiffres) avec tout ce que fournit le chapitre : dropout, batchnorm, AdamW et ton early stopping (20.28) ; une ablation pour savoir ce qui retarde vraiment l'overfitting (20.29) ; un réseau de 30 couches, avec et sans connexions résiduelles (20.30) ; puis le défi (20.31). Ces exercices réutilisent tes fonctions `train_one_epoch_23` et `evaluate_23` (20.23) et ta classe `EarlyStopping` (20.17).

### Ex 20.28 — Dropout, batchnorm, AdamW et early stopping sur Fashion-MNIST 📦 ★★★ ⏱️ 60 min
**Objectif :** entraîner un MLP régularisé sur Fashion-MNIST avec un early stopping qui garde les meilleurs poids.  
**Prérequis :** Ex 20.23 · Ex 20.17 · fiche §20.4.3, §20.4.4, section sur la régularisation · **Fil rouge :** Fashion-MNIST · **Parcours :** R, C

Écris :
- `make_model_28(seed=0)` : un MLP 784-256-256-10 pour Fashion-MNIST, initialisé avec la graine et envoyé sur `DEVICE` : un aplatissement, puis, pour chaque couche cachée, une couche dense, une batchnorm (`nn.BatchNorm1d`), une ReLU et un dropout de 0,3 ; une dernière couche dense vers les 10 logits (la docstring donne l'ordre) ;
- `fit_28(model, max_epochs, patience)` : l'entraînement avec AdamW (learning rate $10^{-3}$, weight decay $10^{-2}$) et un early stopping sur la loss de validation. À chaque epoch (numérotée à partir de 1) : `train_one_epoch_23` sur `train_loader_28(epoch)`, `evaluate_23` sur `VAL_LOADER_28` (tes fonctions de 20.23), puis le `step` de **ta** classe `EarlyStopping` de `mylearn.nn.regularization` ; à chaque amélioration, une **copie profonde** de `model.state_dict()` ; l'entraînement s'arrête quand `step` le dit, ou au bout de `max_epochs` ; à la fin, les meilleurs poids sont rechargés dans `model`. Elle rend l'historique, un dictionnaire de trois listes de même longueur (une valeur par epoch) : `"train_loss"`, `"val_loss"` et `"val_acc"`.

La vérification contrôle l'architecture (270 346 paramètres), lance `fit_28` (au plus 15 epochs en mode rapide, sur 10 000 images ; 40 epochs et 50 000 images en mode complet), contrôle l'arrêt, que le modèle final a bien les poids de la meilleure epoch, et une accuracy de validation de plus de 85 %, puis trace les courbes.

Dans tes notes : à quelle epoch l'early stopping a-t-il gardé les poids ? Que se passerait-il sans le `copy.deepcopy` (essaie) ? Pourquoi la loss d'entraînement reste-t-elle au-dessus de la loss de validation pendant les premières epochs ?

In [ ]:
X_F28, Y_F28 = wb.datasets.load_fashion_mnist("train", normalize=True)      # 60 000 images of clothes, float32 in [0, 1]
N_TRAIN_28 = wb.by_mode(fast=10_000, full=50_000)
TRAIN_SET_28 = TensorDataset(torch.from_numpy(X_F28[:N_TRAIN_28]).unsqueeze(1), torch.from_numpy(Y_F28[:N_TRAIN_28]))
VAL_LOADER_28 = DataLoader(TensorDataset(torch.from_numpy(X_F28[50_000:]).unsqueeze(1), torch.from_numpy(Y_F28[50_000:])),
                           batch_size=1000)                # the last 10 000 images: the validation
MAX_EPOCHS_28, PATIENCE_28 = wb.by_mode(fast=15, full=40), 3


def train_loader_28(epoch):
    """The training mini-batches of the epoch (128 images), shuffled by a generator seeded with the epoch number."""
    return DataLoader(TRAIN_SET_28, batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(epoch))


print(f"Fashion-MNIST: {len(TRAIN_SET_28)} training images, 10 000 for validation · classes: {wb.datasets.FASHION_CLASSES}")

In [ ]:
def make_model_28(seed=0):
    """A MLP 784-256-256-10 for Fashion-MNIST, initialized with the seed and sent to DEVICE: nn.Flatten(), then for each
    hidden layer nn.Linear -> nn.BatchNorm1d -> nn.ReLU -> nn.Dropout(0.3), and a last nn.Linear to the 10 logits."""
    raise NotImplementedError("make_model_28")


def fit_28(model, max_epochs=MAX_EPOCHS_28, patience=PATIENCE_28):
    """Train model with AdamW (lr 1e-3, weight_decay 1e-2) and an early stopping on the validation loss.

    Each epoch: train_one_epoch_23 on train_loader_28(epoch) (epochs numbered from 1), then evaluate_23 on VAL_LOADER_28
    (your functions of 20.23), then the step of YOUR mylearn.nn.regularization.EarlyStopping(patience=patience); a deep
    copy of model.state_dict() is kept at each improvement; the training stops when the step says so, or after
    max_epochs. At the end, the best weights are loaded back into model. Returns the history, a dict of three lists of
    the same length (one value per epoch): "train_loss", "val_loss" and "val_acc".
    """
    raise NotImplementedError("fit_28")

In [ ]:
history_28 = None
with checked("20.28"):
    model_28 = make_model_28(0)
    kinds_28 = [type(m).__name__ for m in model_28.modules() if not list(m.children())]
    verdict("20.28", kinds_28.count("Linear") == 3 and kinds_28.count("BatchNorm1d") == 2 and kinds_28.count("Dropout") == 2 and n_params(model_28) == 270_346,
            "make_model_28 : 3 couches denses, 2 batchnorms, 2 dropouts, " + f"{n_params(model_28):_} paramètres.".replace("_", " "),
            f"make_model_28 contient {kinds_28} et {n_params(model_28)} paramètres : relis la docstring (Linear -> BatchNorm1d -> ReLU -> Dropout, deux fois).")
    started_28 = time.perf_counter()
    history_28 = fit_28(model_28)
    seconds_28 = time.perf_counter() - started_28
    ok_28 = isinstance(history_28, dict) and set(history_28) == {"train_loss", "val_loss", "val_acc"} and len({len(v) for v in history_28.values()}) == 1
    if not ok_28:
        print("❌ Ex 20.28 : fit_28 doit rendre un dictionnaire de trois listes de même longueur : train_loss, val_loss et val_acc.")
        history_28 = None
    else:
        val_28 = np.array(history_28["val_loss"])
        best_28 = int(np.argmin(val_28))
        n_28 = len(val_28)
        stopped_28 = n_28 < MAX_EPOCHS_28
        verdict("20.28", (not stopped_28 and n_28 == MAX_EPOCHS_28) or (stopped_28 and n_28 - 1 - best_28 == PATIENCE_28),
                f"{n_28} epochs en {seconds_28:.0f} s ; meilleure loss de validation à l'epoch {best_28 + 1}" + (f", arrêt {PATIENCE_28} epochs plus tard." if stopped_28 else ", sans que l'early stopping se déclenche."),
                f"{n_28} epochs, meilleure à l'epoch {best_28 + 1} : avec une patience de {PATIENCE_28}, l'arrêt doit venir {PATIENCE_28} epochs après la meilleure (ou au bout de {MAX_EPOCHS_28}).")
        restored_28 = evaluate_23(model_28, VAL_LOADER_28, nn.CrossEntropyLoss(), DEVICE)
        verdict("20.28", abs(restored_28[0] - val_28[best_28]) < 1e-5,
                f"le modèle final a les poids de la meilleure epoch : loss de validation {restored_28[0]:.4f}, accuracy {restored_28[1]:.1%}.",
                f"le modèle final a une loss de validation de {restored_28[0]:.4f}, pas la meilleure ({val_28[best_28]:.4f}) : garde une copie "
                "PROFONDE du state_dict à chaque amélioration, et recharge-la à la fin.")
        verdict("20.28", restored_28[1] > 0.85, f"plus de 85 % en validation ({restored_28[1]:.1%}).",
                f"{restored_28[1]:.1%} en validation : attendu plus de 85 %.")
if history_28 is not None:
    epochs_28 = np.arange(1, len(history_28["val_loss"]) + 1)
    fig_28, (ax_l28, ax_a28) = plt.subplots(1, 2, figsize=(12, 3.8))
    ax_l28.plot(epochs_28, history_28["train_loss"], label="entraînement (avec dropout)")
    ax_l28.plot(epochs_28, history_28["val_loss"], label="validation")
    ax_l28.axvline(int(np.argmin(history_28["val_loss"])) + 1, color="0.6", ls="--", label="poids gardés")
    ax_l28.set_xlabel("epoch")
    ax_l28.set_ylabel("loss")
    ax_l28.legend()
    ax_a28.plot(epochs_28, history_28["val_acc"])
    ax_a28.set_xlabel("epoch")
    ax_a28.set_ylabel("accuracy de validation")
    plt.tight_layout()
    plt.show()

### Ex 20.29 — Ablation : qu'est-ce qui retarde l'overfitting ? 🔬 ★★★ ⏱️ 60 min
**Objectif :** mesurer, par une ablation sur un petit jeu d'entraînement, ce qu'apportent le dropout, le weight decay et la batchnorm.  
**Prérequis :** Ex 20.28 · fiche §20.4.3, §20.4.4, section sur la régularisation (l'ablation) · **Fil rouge :** Fashion-MNIST · **Parcours :** C

Avec seulement 2 000 images d'entraînement, un MLP 784-256-256-10 surapprend vite. Écris `run_29(dropout, weight_decay, batchnorm, seed, epochs=EPOCHS_29)`, qui entraîne ce MLP selon le protocole de la docstring (modèle construit après `torch.manual_seed(seed)`, AdamW, mini-batches de 64 mélangés par un générateur de graine `seed`, pas d'early stopping ; tes fonctions de 20.23) et rend son historique : trois listes de `epochs` valeurs, `"train_loss"`, `"val_loss"` et `"val_acc"`.

La vérification fait un petit contrôle (2 epochs), puis lance l'ablation : cinq configurations (rien ; un dropout de 0,5 ; un weight decay de 1 ; la batchnorm ; les trois ensemble), 2 graines et 25 epochs chacune (3 graines et 40 epochs en mode complet), de 30 s à une minute sur CPU (davantage sur une machine chargée). Elle affiche, en moyenne sur les graines, l'epoch de la meilleure loss de validation, cette meilleure loss, la loss de validation finale, la loss d'entraînement finale et la meilleure accuracy, puis trace les courbes de validation et d'entraînement.

Dans tes notes : quelle technique **retarde** le plus le minimum de la loss de validation ? Laquelle améliore le plus la loss finale ? Que fait la batchnorm : régularise-t-elle, ou accélère-t-elle l'apprentissage (relis l'encadré de la fiche) ? Les écarts d'accuracy entre configurations sont-ils plus grands que les écarts d'une graine à l'autre ? Que faudrait-il pour conclure plus sûrement ?

In [ ]:
TRAIN_SET_29 = TensorDataset(torch.from_numpy(X_F28[:2000]).unsqueeze(1), torch.from_numpy(Y_F28[:2000]))   # 2 000 images: overfitting comes fast
VAL_LOADER_29 = DataLoader(TensorDataset(torch.from_numpy(X_F28[50_000:55_000]).unsqueeze(1), torch.from_numpy(Y_F28[50_000:55_000])),
                           batch_size=1000)
EPOCHS_29, SEEDS_29 = wb.by_mode(fast=25, full=40), wb.by_mode(fast=[0, 1], full=[0, 1, 2])
CONFIGS_29 = {"rien": dict(dropout=0.0, weight_decay=0.0, batchnorm=False),
              "dropout 0,5": dict(dropout=0.5, weight_decay=0.0, batchnorm=False),
              "weight decay 1": dict(dropout=0.0, weight_decay=1.0, batchnorm=False),
              "batchnorm": dict(dropout=0.0, weight_decay=0.0, batchnorm=True),
              "les trois": dict(dropout=0.5, weight_decay=1.0, batchnorm=True)}
print(f"{len(TRAIN_SET_29)} training images · {len(CONFIGS_29)} configurations x {len(SEEDS_29)} seeds x {EPOCHS_29} epochs")

In [ ]:
def run_29(dropout, weight_decay, batchnorm, seed, epochs=EPOCHS_29):
    """Train a MLP 784-256-256-10 on TRAIN_SET_29 and return its history, a dict of three lists of `epochs` values
    ("train_loss", "val_loss", "val_acc"), with train_one_epoch_23 and evaluate_23 (VAL_LOADER_29).

    The model, built after torch.manual_seed(seed): nn.Flatten(), then for each hidden layer nn.Linear, an
    nn.BatchNorm1d if batchnorm, nn.ReLU, an nn.Dropout(dropout) if dropout > 0; a last nn.Linear to the logits; on
    DEVICE. AdamW(lr=1e-3, weight_decay=weight_decay); mini-batches of 64 from ONE DataLoader for all the epochs,
    shuffled by torch.Generator().manual_seed(seed) (so each epoch sees a new order); no early stopping.
    """
    raise NotImplementedError("run_29")

In [ ]:
RESULTS_29 = {}                                  # the trainings of the ablation (torch models of run_29)
with checked("20.29"):
    control_29 = run_29(0.0, 0.0, False, seed=0, epochs=2)
    ok_29 = isinstance(control_29, dict) and all(len(control_29.get(k, [])) == 2 for k in ("train_loss", "val_loss", "val_acc"))
    if not ok_29:
        print("❌ Ex 20.29 : run_29(..., epochs=2) doit rendre un dictionnaire de trois listes de 2 valeurs : train_loss, val_loss et val_acc.")
    else:
        started_29 = time.perf_counter()
        for name_29, config_29 in CONFIGS_29.items():
            RESULTS_29[name_29] = [run_29(**config_29, seed=seed_29) for seed_29 in SEEDS_29]
        print(f"{len(CONFIGS_29) * len(SEEDS_29)} trainings in {time.perf_counter() - started_29:.0f} s (means over the seeds)\n")
        print("configuration      best epoch   min val loss   final val loss   final train loss   best val acc")
        for name_29, runs_29 in RESULTS_29.items():
            val_29 = np.array([r["val_loss"] for r in runs_29])
            print(f"{name_29:16} {np.mean(val_29.argmin(axis=1) + 1):12.1f} {val_29.min(axis=1).mean():14.3f} "
                  f"{val_29[:, -1].mean():16.3f} {np.mean([r['train_loss'][-1] for r in runs_29]):18.3f} "
                  f"{np.mean([max(r['val_acc']) for r in runs_29]):14.3f}")
        none_29 = np.array([r["val_loss"] for r in RESULTS_29["rien"]])
        verdict("20.29", none_29[:, -1].mean() > none_29.min(axis=1).mean() + 0.05,
                "sans régularisation, la loss de validation remonte nettement après son minimum : l'overfitting est là.",
                "sans régularisation, la loss de validation ne remonte pas : vérifie run_29 (2 000 images, pas d'early stopping).")
if RESULTS_29:
    fig_29, axes_29 = plt.subplots(1, 2, figsize=(14, 4))
    for name_29, runs_29 in RESULTS_29.items():
        epochs_29 = np.arange(1, EPOCHS_29 + 1)
        line_29, = axes_29[0].plot(epochs_29, np.mean([r["val_loss"] for r in runs_29], axis=0), label=name_29)
        axes_29[1].plot(epochs_29, np.mean([r["train_loss"] for r in runs_29], axis=0), color=line_29.get_color())
    axes_29[0].set_title("loss de validation (moyenne sur les graines)")
    axes_29[1].set_title("loss d'entraînement (mesurée pendant l'epoch)")
    for ax in axes_29:
        ax.set_xlabel("epoch")
        ax.set_yscale("log")
    axes_29[0].legend()
    plt.tight_layout()
    plt.show()

### Ex 20.30 — Un réseau de 30 couches : connexions résiduelles et LayerNorm 🔬 ★★★ ⏱️ 45 min
**Objectif :** mesurer sur un MLP de 30 couches ce que changent les connexions résiduelles et la layer norm, au gradient et à l'apprentissage.  
**Prérequis :** Ex 20.23 · ✏️ 20.7 · ch. 18 (gradient évanescent) · fiche §20.4.8, section sur les connexions résiduelles · **Fil rouge :** MNIST · **Parcours :** C

Écris la classe `Deep30(nn.Module)` (la docstring donne le contrat : une couche d'entrée `self.inp`, `depth` blocs de largeur `width`, une couche de sortie `self.out` ; chaque bloc calcule $h = \mathrm{ReLU}(\text{linear}(\text{norm}(x)))$, avec une `nn.LayerNorm` si `norm`, puis $x \leftarrow x + h$ si `residual`, sinon $x \leftarrow h$), et `grad_ratio_30(model, X, y)` : après **une** passe arrière de la cross-entropy sur `(X, y)`, la norme du gradient du poids de la couche d'entrée divisée par celle du gradient du poids de la couche de sortie.

La vérification contrôle un petit `Deep30`, puis, pour quatre réseaux de 30 blocs de 64 neurones (sans rien ; avec la layer norm seule ; avec les connexions résiduelles seules ; avec les deux), mesure ce rapport à l'initialisation, puis entraîne chaque réseau 2 epochs sur 5 000 images MNIST (`train_30`, fourni) et affiche la loss et l'accuracy.

Dans tes notes : relie le rapport des gradients du réseau sans connexion résiduelle au calcul de ✏️ 20.7 f. Pourquoi le réseau résiduel sans layer norm a-t-il un rapport plus proche de 1, mais des gradients énormes (essaie d'afficher la norme du gradient de sortie) ? Que fait la layer norm quand elle est seule ? Quelle combinaison retrouves-tu dans chaque bloc d'un Transformer ?

In [ ]:
X_30, Y_30 = wb.datasets.load_mnist("train", n=5000, flatten=True, normalize=True)
X_30, Y_30 = torch.from_numpy(X_30).to(DEVICE), torch.from_numpy(Y_30).to(DEVICE)       # 5 000 images (5000, 784)
EPOCHS_30 = wb.by_mode(fast=2, full=5)
CONFIGS_30 = {"plain": dict(residual=False, norm=False), "LayerNorm only": dict(residual=False, norm=True),
              "residual": dict(residual=True, norm=False), "residual + LayerNorm": dict(residual=True, norm=True)}


def train_30(model, epochs=EPOCHS_30, seed=0):
    """Adam (lr 1e-3), mini-batches of 64 of (X_30, Y_30); returns (loss, accuracy) on the 5 000 images after training."""
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    generator = torch.Generator().manual_seed(seed)
    model.train()
    for _ in range(epochs):
        for idx in torch.randperm(len(X_30), generator=generator).split(64):
            loss = nn.functional.cross_entropy(model(X_30[idx]), Y_30[idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        logits = model(X_30)
        return nn.functional.cross_entropy(logits, Y_30).item(), (logits.argmax(dim=1) == Y_30).float().mean().item()

In [ ]:
class Deep30(nn.Module):
    """A deep MLP for MNIST: an input layer nn.Linear(784, width), then `depth` blocks, then nn.Linear(width, 10).

    Block k: h = relu(linear_k(norm_k(x))), where norm_k is an nn.LayerNorm(width) if norm, else nn.Identity(), and
    linear_k an nn.Linear(width, width); then x = x + h if residual, else x = h. The input layer is stored in
    self.inp, the blocks in an nn.ModuleList, the output layer in self.out.
    """

    def __init__(self, depth=30, width=64, residual=False, norm=False):
        super().__init__()
        raise NotImplementedError("Deep30.__init__")

    def forward(self, x):
        raise NotImplementedError("Deep30.forward")


def grad_ratio_30(model, X, y):
    """After ONE backward pass of the cross-entropy of model on (X, y): the norm of the gradient of the weight of the
    input layer (model.inp) divided by the norm of the gradient of the weight of the output layer (model.out)."""
    raise NotImplementedError("grad_ratio_30")

In [ ]:
RESULTS_30 = {}
with checked("20.30"):
    torch.manual_seed(0)
    probe_30 = Deep30(depth=3, width=8, residual=True, norm=True)
    names_30 = [type(m).__name__ for m in probe_30.modules() if not list(m.children())]
    verdict("20.30", names_30.count("Linear") == 5 and names_30.count("LayerNorm") == 3,
            "Deep30(depth=3, norm=True) : 5 couches denses et 3 layer norms.",
            f"Deep30(depth=3, norm=True) contient {names_30} : attendu 5 nn.Linear (entrée, 3 blocs, sortie) et 3 nn.LayerNorm.")
    x_p30 = torch.randn(4, 784)
    with torch.no_grad():
        plain_p30 = Deep30(depth=3, width=8, residual=False, norm=True)
        plain_p30.load_state_dict(probe_30.state_dict())
    verdict("20.30", not torch.allclose(probe_30(x_p30), plain_p30(x_p30)), "residual change le calcul (x + h).",
            "les versions avec et sans connexion résiduelle donnent la même sortie : applique x = x + h quand residual.")
    for name_30, config_30 in CONFIGS_30.items():
        torch.manual_seed(30)
        model_30 = Deep30(**config_30).to(DEVICE)
        ratio_30 = grad_ratio_30(model_30, X_30[:256], Y_30[:256])
        loss_30, acc_30 = train_30(model_30)
        RESULTS_30[name_30] = (ratio_30, loss_30, acc_30)
    print("30 blocks of 64 neurons     grad ratio first/last layer at init   loss after training   accuracy")
    for name_30, (ratio_30, loss_30, acc_30) in RESULTS_30.items():
        print(f"{name_30:27} {ratio_30:36.2e} {loss_30:21.3f} {acc_30:10.3f}")
    verdict("20.30", RESULTS_30["plain"][0] < 1e-6 and RESULTS_30["plain"][2] < 0.3,
            "sans connexion résiduelle : le gradient n'atteint presque plus la première couche, et le réseau n'apprend pas.",
            "le réseau sans connexion résiduelle apprend, ou son gradient atteint la première couche : vérifie Deep30 (30 blocs, ReLU, pas de raccourci).")
    verdict("20.30", RESULTS_30["residual + LayerNorm"][2] > 0.85,
            "avec connexions résiduelles et layer norm : le même réseau de 30 couches apprend.",
            "avec connexions résiduelles et layer norm, le réseau n'apprend pas : vérifie x = x + h et la place de la layer norm.")

### Ex 20.31 — Défi Fashion-MNIST : au moins 90 % avec un MLP de 300 k paramètres au plus 🏆 ★★★★ ⏱️ 120 min
**Objectif :** atteindre 90 % sur le test de Fashion-MNIST avec un MLP PyTorch de 300 000 paramètres au plus, en 3 minutes d'entraînement.  
**Prérequis :** Ex 20.28 · Ex 20.29 · ch. 19 (plannings) · fiche, sections sur la régularisation · **Fil rouge :** Fashion-MNIST · **Parcours :** C

Les 60 000 images d'entraînement de Fashion-MNIST (`X_TRAIN_31`, un tenseur `(60000, 784)` de valeurs dans $[0, 1]$, et leurs labels), et un **test caché** de 10 000 images. Un MLP simple (784-256-10, Adam, 20 epochs) plafonne vers 89 % sur le test. Écris `train_31(X, y)`, qui entraîne un MLP avec PyTorch et rend le modèle entraîné : un `nn.Module` qui transforme des images `(N, 784)` en 10 logits. Tu choisis l'architecture, la régularisation, l'optimiseur, le planning, la taille des mini-batches.

Règles : un **MLP** (seulement des couches denses, des normalisations, du dropout et des activations : pas de convolution) d'au plus **300 000 paramètres** ; au plus **3 minutes** d'entraînement sur ta machine ; le défi est réussi avec une accuracy d'au moins **90 %** sur le test caché ; palier 🌟 : au moins 91 %. Règle d'honneur : `train_31` entraîne vraiment le réseau (pas de poids préparés à l'avance), ne lit jamais `X_HIDDEN_31`, et n'utilise pas le test caché pour régler ta méthode : garde, par exemple, les 10 000 dernières images d'entraînement comme validation pendant tes essais, comme en 20.28. La vérification ne le révèle qu'une fois par session et par architecture (et seulement si ton réseau dépasse 90 % sur des images d'entraînement), et fige le résultat.

Dans tes notes : ta démarche (ce que tu as essayé, dans quel ordre, avec quelle validation), et ce qui a le plus compté : la taille du réseau, la batchnorm, le dropout, le planning, le nombre d'epochs ?

In [ ]:
X_TRAIN_31, Y_TRAIN_31 = (torch.from_numpy(a) for a in wb.datasets.load_fashion_mnist("train", flatten=True, normalize=True))
X_HIDDEN_31, Y_HIDDEN_31 = (torch.from_numpy(a) for a in wb.datasets.load_fashion_mnist("test", flatten=True, normalize=True))   # do not look
MAX_PARAMS_31, TIME_LIMIT_31 = 300_000, 180.0
ALLOWED_31 = (nn.Linear, nn.BatchNorm1d, nn.LayerNorm, nn.RMSNorm, nn.Dropout, nn.Flatten, nn.Identity, nn.ReLU, nn.LeakyReLU,
              nn.GELU, nn.SiLU, nn.ELU, nn.Tanh, nn.Sigmoid, nn.Mish, nn.Softplus, nn.PReLU)
print("train:", tuple(X_TRAIN_31.shape), "· hidden test:", tuple(X_HIDDEN_31.shape))

In [ ]:
def train_31(X, y):
    """Train a MLP on the Fashion-MNIST images X (a float32 tensor (60000, 784), values in [0, 1]) and their labels y
    (int64), with PyTorch, and return the trained model: an nn.Module that maps (N, 784) images to (N, 10) logits.

    Only dense layers, normalizations, dropout and activations (no convolution); at most 300 000 parameters; at most
    3 minutes of training.
    """
    raise NotImplementedError("train_31")

In [ ]:
with checked("20.31"):
    started_31 = time.perf_counter()
    model_31 = train_31(X_TRAIN_31, Y_TRAIN_31)
    seconds_31 = time.perf_counter() - started_31
    if not isinstance(model_31, nn.Module):
        print(f"❌ Ex 20.31 : train_31 doit rendre le modèle entraîné, un nn.Module ; reçu {type(model_31).__name__}.")
    else:
        leaves_31 = [m for m in model_31.modules() if not list(m.children())]
        bad_31 = sorted({type(m).__name__ for m in leaves_31 if not isinstance(m, ALLOWED_31)})
        ok_31 = not bad_31
        verdict("20.31", ok_31, "un MLP : des couches denses, des normalisations, du dropout et des activations.",
                f"des couches interdites dans un MLP : {bad_31} (pas de convolution, pas de récurrence).")
        count_31 = n_params(model_31)
        verdict("20.31", count_31 <= MAX_PARAMS_31, f"{count_31:_} paramètres (au plus 300 000).".replace("_", " "),
                f"{count_31:_} paramètres : le défi en permet au plus 300 000.".replace("_", " "))
        verdict("20.31", seconds_31 <= TIME_LIMIT_31, f"entraînement en {seconds_31:.0f} s (au plus {TIME_LIMIT_31:.0f} s).",
                f"entraînement en {seconds_31:.0f} s : le défi en permet au plus {TIME_LIMIT_31:.0f} (moins d'epochs, des mini-batches plus grands).")
        ok_31 = ok_31 and count_31 <= MAX_PARAMS_31 and seconds_31 <= TIME_LIMIT_31
        if ok_31:
            model_31 = model_31.to(DEVICE).eval()
            with torch.no_grad():
                train_acc_31 = (model_31(X_TRAIN_31[:10_000].to(DEVICE)).argmax(dim=1).cpu() == Y_TRAIN_31[:10_000]).float().mean().item()
            print(f"accuracy on 10 000 training images: {train_acc_31:.4f}")
            ok_31 = train_acc_31 >= 0.90
            if not ok_31:
                print("❌ Ex 20.31 : moins de 90 % sur des images d'entraînement : le test caché n'est pas révélé ; améliore d'abord l'entraînement.")
        if ok_31:
            REVEALED_31 = globals().get("REVEALED_31") or {}
            signature_31 = tuple((type(m).__name__, tuple(p.shape for p in m.parameters(recurse=False))) for m in leaves_31)
            if signature_31 not in REVEALED_31:                    # revealed once per architecture and per session (an honour rule)
                with torch.no_grad():
                    REVEALED_31[signature_31] = ((model_31(X_HIDDEN_31.to(DEVICE)).argmax(dim=1).cpu() == Y_HIDDEN_31).float().mean().item(),
                                                 round(seconds_31))
            test_31, time_31 = REVEALED_31[signature_31]
            print(f"hidden test: {test_31:.4f} (revealed once per session for this architecture, trained in {time_31} s; "
                  "a second run of the same architecture shows the same frozen result)")
            verdict("20.31", test_31 >= 0.90,
                    "défi réussi : au moins 90 % sur les 10 000 images du test caché, avec un MLP de 300 000 paramètres au plus."
                    + (" 🌟 Palier atteint : au moins 91 %." if test_31 >= 0.91 else ""),
                    "pas encore : il faut au moins 90 % sur le test caché.")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu prévoir la forme d'un tenseur après une couche dense, un aplatissement ou un pooling, et choisir la sortie et la loss d'un réseau ?
2. Sais-tu calculer à la main un dropout inversé et une batchnorm, et dire ce que changent `model.train()` et `model.eval()` ?
3. Sais-tu écrire une boucle d'entraînement PyTorch complète et expliquer chacune de ses lignes ?

**Pour aller plus loin** : le tutoriel « Learn the Basics » de PyTorch et *Dive into Deep Learning*, cités dans la fiche. La suite : le ch. 21, les réseaux de convolution, en PyTorch.